> **Versión GitHub reproducible.** Esta copia no modifica el notebook original de Google Drive. Las rutas personales de Drive se han sustituido por rutas del repositorio. Los datos brutos de gran tamaño no incluidos en GitHub deben descargarse según `docs/DATA_SOURCES.md`.


In [ ]:
# ============================================================
# CONFIGURACIÓN DE RUTAS - VERSIÓN GITHUB / GOOGLE COLAB
# ============================================================
from pathlib import Path
import os
import subprocess

ROOT = Path("/content/TFM_Rendimiento_cultivos")

# Si el notebook se abre directamente desde GitHub en Colab,
# descargamos automáticamente el repositorio.
if not ROOT.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/ritaridh/tfm-crop-yield-prediction.git",
            str(ROOT),
        ],
        check=True,
    )

os.chdir(ROOT)

RUTA_TFM = str(ROOT)
RUTA_BASE = str(ROOT)
BASE = str(ROOT)
BASE_DIR = ROOT
RUTA_ORIGINALES = str(ROOT / "01_Datos_originales")
RUTA_EUROCROPS = str(ROOT / "01_Datos_originales" / "EuroCrops")
RUTA_MAPA = str(ROOT / "01_Datos_originales" / "MAPA")
RUTA_SENTINEL = str(ROOT / "01_Datos_originales" / "Sentinel2")
RUTA_PROCESADOS = str(ROOT / "02_Datos_procesados")
RUTA_RESULTADOS = str(ROOT / "05_Resultados")
PROC = RUTA_PROCESADOS
RESULTADOS = RUTA_RESULTADOS

Path(RUTA_PROCESADOS).mkdir(parents=True, exist_ok=True)
Path(RUTA_RESULTADOS).mkdir(parents=True, exist_ok=True)

print("Repositorio:", ROOT)


In [ ]:
# ============================================================
# PASO PREVIO - CONECTAR GOOGLE DRIVE
# Y DEFINIR LA CARPETA PRINCIPAL DEL TFM
# ============================================================

import os

# 1. Montar Google Drive

# 2. Definir carpeta principal del TFM
ROOT = "/content/TFM_Rendimiento_cultivos"

# 3. Comprobar que existe
print("\nCarpeta principal del TFM:")
print(ROOT)

print("\nExiste:", os.path.exists(ROOT))

# 4. Mostrar contenido principal
if os.path.exists(ROOT):
    print("\nContenido de la carpeta:")
    for nombre in sorted(os.listdir(ROOT)):
        print("-", nombre)
else:
    print("\nERROR: No se encuentra la carpeta del TFM.")

In [ ]:
# ============================================================
# PASO 2 - INSTALAR LIBRERÍAS NECESARIAS PARA LA DEMO
# ============================================================

!pip install -q gradio xgboost joblib

print("PASO 2 FINALIZADO")

In [ ]:
# ============================================================
# PASO 3 - COMPROBAR DATASET DEFINITIVO DEL TFM
# ============================================================

import os
import pandas as pd
import numpy as np

# Ruta principal ya definida anteriormente:
# ROOT = "/content/TFM_Rendimiento_cultivos"

F_DATASET = (
    f"{ROOT}/02_Datos_procesados/"
    "dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

print("=" * 80)
print("PASO 3 - COMPROBACIÓN DEL DATASET DEFINITIVO")
print("=" * 80)

print("\nArchivo:")
print(F_DATASET)

print("\nExiste:", os.path.exists(F_DATASET))

if not os.path.exists(F_DATASET):
    raise FileNotFoundError(
        "No se encuentra dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
    )

# Cargar dataset
df = pd.read_csv(F_DATASET)

print("\nDimensiones:")
print(df.shape)

print("\nAños disponibles:")
print(sorted(df["ano"].unique()))

print("\nCultivos:")
print(df["cultivo"].value_counts())

print("\nNúmero de provincias:")
print(df["provincia_estandar"].nunique())

print("\nPrimeras columnas:")
print(df.columns[:15].tolist())

print("\nPASO 3 FINALIZADO")

In [ ]:
# ============================================================
# PASO 4 - DEFINIR LOS PREDICTORES DEL MODELO
# ============================================================

TARGET = "rendimiento_kg_ha"

# Columnas que NO entran como predictores
EXCLUIR = [
    TARGET,
    "ano",
    "provincia_estandar"
]

# Predictores finales
features = [
    c for c in df.columns
    if c not in EXCLUIR
]

# Variable categórica
cat_cols = ["cultivo"]

# Variables numéricas
num_cols = [
    c for c in features
    if c not in cat_cols
]

print("=" * 80)
print("PASO 4 - PREDICTORES DEL MODELO")
print("=" * 80)

print("\nTotal predictores:", len(features))
print("Numéricos:", len(num_cols))
print("Categóricos:", len(cat_cols))

print("\nVariable categórica:")
print(cat_cols)

print("\nListado completo de predictores:")
for i, variable in enumerate(features, 1):
    print(f"{i:02d}. {variable}")

# Controles obligatorios
assert len(features) == 48, (
    f"ERROR: se esperaban 48 predictores y se han encontrado {len(features)}"
)

assert len(num_cols) == 47, (
    f"ERROR: se esperaban 47 variables numéricas y se han encontrado {len(num_cols)}"
)

assert cat_cols == ["cultivo"], (
    "ERROR: la variable categórica esperada es 'cultivo'"
)

print("\nCONTROLES CORRECTOS")
print("PASO 4 FINALIZADO")

In [ ]:
# ============================================================
# PASO 5 - CREAR ESTRUCTURA DE CARPETAS PARA LA DEMO
# ============================================================

import os

# Carpeta principal de la demo
DEMO_DIR = f"{ROOT}/06_Demo_UI"

# Subcarpetas
MODELOS_DIR = f"{DEMO_DIR}/modelos"
CONFIG_DIR = f"{DEMO_DIR}/configuracion"
RESULTADOS_DEMO_DIR = f"{DEMO_DIR}/resultados_demo"

# Crear carpetas
for carpeta in [
    DEMO_DIR,
    MODELOS_DIR,
    CONFIG_DIR,
    RESULTADOS_DEMO_DIR
]:
    os.makedirs(carpeta, exist_ok=True)

print("=" * 80)
print("PASO 5 - ESTRUCTURA DE LA DEMO")
print("=" * 80)

print("\nCarpeta principal:")
print(DEMO_DIR)

print("\nSubcarpetas:")
for carpeta in [
    MODELOS_DIR,
    CONFIG_DIR,
    RESULTADOS_DEMO_DIR
]:
    print(f"- {carpeta} -> Existe: {os.path.exists(carpeta)}")

# Controles
assert os.path.exists(DEMO_DIR)
assert os.path.exists(MODELOS_DIR)
assert os.path.exists(CONFIG_DIR)
assert os.path.exists(RESULTADOS_DEMO_DIR)

print("\nESTRUCTURA CREADA CORRECTAMENTE")
print("PASO 5 FINALIZADO")

In [ ]:
# ============================================================
# PASO 6 - SEPARACIÓN TEMPORAL DEL DATASET
# ============================================================

train = df[df["ano"] <= 2022].copy()
test = df[df["ano"] == 2023].copy()

print("=" * 80)
print("PASO 6 - SEPARACIÓN TEMPORAL")
print("=" * 80)

print("\nTRAIN / DESARROLLO")
print("Años:", sorted(train["ano"].unique()))
print("Observaciones:", len(train))

print("\nTEST INDEPENDIENTE")
print("Años:", sorted(test["ano"].unique()))
print("Observaciones:", len(test))

print("\nDistribución por cultivo en TRAIN:")
print(train["cultivo"].value_counts())

print("\nDistribución por cultivo en TEST:")
print(test["cultivo"].value_counts())

# Controles
assert len(train) == 572, (
    f"ERROR: se esperaban 572 observaciones de train y hay {len(train)}"
)

assert len(test) == 95, (
    f"ERROR: se esperaban 95 observaciones de test y hay {len(test)}"
)

assert sorted(train["ano"].unique()) == [2017, 2018, 2019, 2020, 2021, 2022]

assert sorted(test["ano"].unique()) == [2023]

print("\nSEPARACIÓN TEMPORAL CORRECTA")
print("PASO 6 FINALIZADO")

In [ ]:
# ============================================================
# PASO 7 - ENTRENAR Y GUARDAR LOS MODELOS DEFINITIVOS
# ============================================================

import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone
from xgboost import XGBRegressor

# ------------------------------------------------------------
# 1. Datos de entrenamiento
# ------------------------------------------------------------

X_train = train[features]
y_train = train[TARGET]

# ------------------------------------------------------------
# 2. Preprocesamiento utilizado en el modelo definitivo
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            cat_cols
        )
    ]
)

# ------------------------------------------------------------
# 3. Random Forest optimizado
# ------------------------------------------------------------

rf = RandomForestRegressor(
    n_estimators=800,
    max_depth=15,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features="sqrt",
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# 4. XGBoost optimizado
# ------------------------------------------------------------

xgb = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.08,
    min_child_weight=3,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# 5. Pipelines completos
# ------------------------------------------------------------

pipe_rf = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("modelo", rf)
])

pipe_xgb = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("modelo", xgb)
])

# ------------------------------------------------------------
# 6. Entrenamiento
# ------------------------------------------------------------

print("Entrenando Random Forest...")
pipe_rf.fit(X_train, y_train)

print("Entrenando XGBoost...")
pipe_xgb.fit(X_train, y_train)

# ------------------------------------------------------------
# 7. Guardar modelos
# ------------------------------------------------------------

F_RF = f"{MODELOS_DIR}/RF_tuned_TFM.joblib"
F_XGB = f"{MODELOS_DIR}/XGB_tuned_TFM.joblib"

joblib.dump(pipe_rf, F_RF)
joblib.dump(pipe_xgb, F_XGB)

print("\nModelos entrenados y guardados correctamente.")
print("Random Forest:", F_RF)
print("XGBoost:", F_XGB)

print("\nPASO 7 FINALIZADO")

In [ ]:
# ============================================================
# PASO 8 - VALORES DE REFERENCIA Y RANGOS DE ENTRENAMIENTO
# ============================================================

# Medianas para inicializar la interfaz
medianas_train = train[num_cols].median()

# Rangos observados para detectar extrapolación
minimos_train = train[num_cols].min()
maximos_train = train[num_cols].max()

# Guardar configuración básica
rangos_train = pd.DataFrame({
    "variable": num_cols,
    "mediana": [medianas_train[c] for c in num_cols],
    "minimo": [minimos_train[c] for c in num_cols],
    "maximo": [maximos_train[c] for c in num_cols]
})

F_RANGOS = f"{CONFIG_DIR}/rangos_entrenamiento_2017_2022.csv"

rangos_train.to_csv(
    F_RANGOS,
    index=False
)

print("Rangos y medianas preparados.")
print(F_RANGOS)

print("\nPASO 8 FINALIZADO")

In [ ]:
# ============================================================
# PASO 9 - FUNCIÓN DE PREDICCIÓN PARA LA DEMO
# ============================================================

import numpy as np
import pandas as pd
import joblib

# ------------------------------------------------------------
# 1. Cargar los modelos guardados
# ------------------------------------------------------------

modelo_rf = joblib.load(
    f"{MODELOS_DIR}/RF_tuned_TFM.joblib"
)

modelo_xgb = joblib.load(
    f"{MODELOS_DIR}/XGB_tuned_TFM.joblib"
)

# ------------------------------------------------------------
# 2. Función principal de predicción
# ------------------------------------------------------------

def predecir_rendimiento(cultivo, valores_usuario):
    """
    Predice el rendimiento agrícola a partir de los valores
    introducidos por el usuario.

    Parámetros
    ----------
    cultivo : str
        'barley' o 'common_soft_wheat'

    valores_usuario : dict
        Diccionario con los 47 predictores numéricos.

    Devuelve
    --------
    dict
        Predicciones RF, XGBoost y ensemble 50/50,
        además de posibles variables fuera del rango
        observado durante el entrenamiento.
    """

    # --------------------------------------------------------
    # Comprobar cultivo
    # --------------------------------------------------------

    cultivos_validos = [
        "barley",
        "common_soft_wheat"
    ]

    if cultivo not in cultivos_validos:
        raise ValueError(
            "Cultivo no válido. Debe ser 'barley' "
            "o 'common_soft_wheat'."
        )

    # --------------------------------------------------------
    # Construir observación
    # --------------------------------------------------------

    fila = {"cultivo": cultivo}

    for variable in num_cols:

        if variable not in valores_usuario:
            raise ValueError(
                f"Falta la variable: {variable}"
            )

        fila[variable] = float(
            valores_usuario[variable]
        )

    X_usuario = pd.DataFrame(
        [fila],
        columns=features
    )

    # --------------------------------------------------------
    # Detectar valores fuera del dominio de entrenamiento
    # --------------------------------------------------------

    fuera_rango = []

    for variable in num_cols:

        valor = X_usuario.iloc[0][variable]

        minimo = minimos_train[variable]
        maximo = maximos_train[variable]

        if valor < minimo or valor > maximo:

            fuera_rango.append({
                "variable": variable,
                "valor": float(valor),
                "min_train": float(minimo),
                "max_train": float(maximo)
            })

    # --------------------------------------------------------
    # Predicciones
    # --------------------------------------------------------

    pred_rf = float(
        modelo_rf.predict(X_usuario)[0]
    )

    pred_xgb = float(
        modelo_xgb.predict(X_usuario)[0]
    )

    # Ensemble 50 % RF + 50 % XGBoost
    pred_ensemble = (
        0.50 * pred_rf +
        0.50 * pred_xgb
    )

    # Evitar resultados físicamente negativos
    pred_rf = max(0.0, pred_rf)
    pred_xgb = max(0.0, pred_xgb)
    pred_ensemble = max(0.0, pred_ensemble)

    # --------------------------------------------------------
    # Resultado
    # --------------------------------------------------------

    return {

        "cultivo": cultivo,

        "pred_RF_kg_ha":
            round(pred_rf, 1),

        "pred_XGB_kg_ha":
            round(pred_xgb, 1),

        "pred_ensemble_kg_ha":
            round(pred_ensemble, 1),

        "variables_fuera_rango":
            fuera_rango,

        "n_variables_fuera_rango":
            len(fuera_rango)
    }


print("Función de predicción creada correctamente.")
print("PASO 9 FINALIZADO")

In [ ]:
# ============================================================
# PASO 10 - PRUEBA DE LA FUNCIÓN DE PREDICCIÓN
# ============================================================

# Crear una observación de prueba utilizando
# las medianas del conjunto de entrenamiento
valores_prueba = {
    variable: float(medianas_train[variable])
    for variable in num_cols
}

# Probar primero con cebada
resultado_prueba = predecir_rendimiento(
    cultivo="barley",
    valores_usuario=valores_prueba
)

print("=" * 80)
print("PASO 10 - PRUEBA DE PREDICCIÓN")
print("=" * 80)

print("\nCultivo: Cebada")

print(
    "Random Forest:",
    resultado_prueba["pred_RF_kg_ha"],
    "kg/ha"
)

print(
    "XGBoost:",
    resultado_prueba["pred_XGB_kg_ha"],
    "kg/ha"
)

print(
    "Ensemble RF + XGB:",
    resultado_prueba["pred_ensemble_kg_ha"],
    "kg/ha"
)

print(
    "\nVariables fuera del rango:",
    resultado_prueba["n_variables_fuera_rango"]
)

print("\nPASO 10 FINALIZADO")

In [ ]:
# ============================================================
# PASO 11 - VALORES DE REFERENCIA POR PROVINCIA Y CULTIVO
# ============================================================

# Medianas históricas de entrenamiento para cada
# combinación provincia + cultivo
referencias_provincia_cultivo = (
    train
    .groupby(
        ["provincia_estandar", "cultivo"]
    )[num_cols]
    .median()
    .reset_index()
)

# Lista de provincias disponibles
provincias_demo = sorted(
    train["provincia_estandar"]
    .dropna()
    .unique()
)

# Guardar para la aplicación
F_REFERENCIAS = (
    f"{CONFIG_DIR}/"
    "referencias_provincia_cultivo_2017_2022.csv"
)

referencias_provincia_cultivo.to_csv(
    F_REFERENCIAS,
    index=False
)

print("=" * 80)
print("PASO 11 - REFERENCIAS PROVINCIA + CULTIVO")
print("=" * 80)

print("\nNúmero de provincias:")
print(len(provincias_demo))

print("\nCombinaciones provincia + cultivo:")
print(len(referencias_provincia_cultivo))

print("\nEjemplo de provincias:")
print(provincias_demo[:10])

print("\nArchivo guardado:")
print(F_REFERENCIAS)

print("\nPASO 11 FINALIZADO")

In [ ]:
# ============================================================
# PASO 12 - INSTALAR E IMPORTAR GRADIO
# ============================================================

!pip install -q gradio

import gradio as gr

print("Versión de Gradio:", gr.__version__)
print("PASO 12 FINALIZADO")

In [ ]:
# ============================================================
# PASO 13 - FUNCIÓN PARA CARGAR VALORES DE REFERENCIA
# POR PROVINCIA Y CULTIVO
# ============================================================

def obtener_referencias(provincia, cultivo):
    """
    Devuelve los 47 valores numéricos medianos históricos
    para una combinación provincia + cultivo.
    """

    fila = referencias_provincia_cultivo[
        (referencias_provincia_cultivo["provincia_estandar"] == provincia) &
        (referencias_provincia_cultivo["cultivo"] == cultivo)
    ]

    if fila.empty:
        # Si no existe esa combinación concreta,
        # utilizar medianas generales del entrenamiento
        return [
            float(medianas_train[variable])
            for variable in num_cols
        ]

    return [
        float(fila.iloc[0][variable])
        for variable in num_cols
    ]


print("Función de referencias creada correctamente.")
print("PASO 13 FINALIZADO")

In [ ]:
# ============================================================
# PASO 14 - CONSTRUIR LA INTERFAZ VISUAL DE LA DEMO
# ============================================================

import gradio as gr

# ------------------------------------------------------------
# 1. AGRUPAR VARIABLES POR BLOQUES
# ------------------------------------------------------------

variables_suelo = [
    "clay_pct_0_30cm",
    "sand_pct_0_30cm",
    "soc_g_kg_0_30cm",
    "ph_0_30cm",
    "bdod_g_cm3_0_30cm"
]

variables_sentinel = [
    c for c in num_cols
    if c.startswith(("NDVI_", "EVI_", "NDMI_"))
]

variables_eurocrops = [
    "n_recintos",
    "superficie_eurocrops_ha",
    "superficie_media_recinto_ha"
]

variables_era5 = [
    c for c in num_cols
    if (
        c.startswith("temp_media_")
        or c.startswith("temp_rocio_")
        or c.startswith("precipitacion_")
        or c.startswith("radiacion_")
    )
]

# Orden de todos los campos numéricos de la UI
variables_ui = (
    variables_era5
    + variables_suelo
    + variables_sentinel
    + variables_eurocrops
)

# ------------------------------------------------------------
# 2. ETIQUETAS AMIGABLES
# ------------------------------------------------------------

def etiqueta_variable(variable):

    etiquetas = {
        "clay_pct_0_30cm": "Arcilla 0–30 cm (%)",
        "sand_pct_0_30cm": "Arena 0–30 cm (%)",
        "soc_g_kg_0_30cm": "Carbono orgánico del suelo (g/kg)",
        "ph_0_30cm": "pH del suelo",
        "bdod_g_cm3_0_30cm": "Densidad aparente (g/cm³)",
        "n_recintos": "Número de recintos agrícolas",
        "superficie_eurocrops_ha": "Superficie del cultivo (ha)",
        "superficie_media_recinto_ha": "Superficie media por recinto (ha)"
    }

    if variable in etiquetas:
        return etiquetas[variable]

    nombre = variable

    reemplazos = [
        ("temp_media_c_", "Temperatura media (°C) — "),
        ("temp_rocio_c_", "Temperatura de rocío (°C) — "),
        ("precipitacion_mm_", "Precipitación (mm) — "),
        ("radiacion_mj_m2_", "Radiación (MJ/m²) — "),
        ("NDVI_", "NDVI — "),
        ("EVI_", "EVI — "),
        ("NDMI_", "NDMI — "),
        ("ene_feb", "Enero–Febrero"),
        ("jan_feb", "Enero–Febrero"),
        ("mar_abr", "Marzo–Abril"),
        ("mar_apr", "Marzo–Abril"),
        ("may_jun", "Mayo–Junio"),
        ("jul_ago", "Julio–Agosto"),
        ("sep_dic", "Septiembre–Diciembre"),
        ("nov_dec", "Noviembre–Diciembre"),
        ("campana", "Campaña completa"),
        ("jul", "Julio")
    ]

    for antiguo, nuevo in reemplazos:
        nombre = nombre.replace(antiguo, nuevo)

    return nombre

# ------------------------------------------------------------
# 3. FUNCIÓN PARA CARGAR VALORES DE REFERENCIA EN LA UI
# ------------------------------------------------------------

def cargar_referencias_ui(provincia, cultivo):

    valores_num = obtener_referencias(
        provincia,
        cultivo
    )

    mapa_valores = dict(
        zip(num_cols, valores_num)
    )

    return [
        mapa_valores[var]
        for var in variables_ui
    ]

# ------------------------------------------------------------
# 4. FUNCIÓN DE PREDICCIÓN CON FORMATO PARA LA UI
# ------------------------------------------------------------

def predecir_desde_ui(cultivo, provincia, *valores):

    datos_usuario = dict(
        zip(variables_ui, valores)
    )

    resultado = predecir_rendimiento(
        cultivo=cultivo,
        valores_usuario=datos_usuario
    )

    pred_ensemble = resultado["pred_ensemble_kg_ha"]
    pred_rf = resultado["pred_RF_kg_ha"]
    pred_xgb = resultado["pred_XGB_kg_ha"]

    pred_t_ha = pred_ensemble / 1000

    n_fuera = resultado["n_variables_fuera_rango"]

    # --------------------------------------------------------
    # Mensaje de dominio
    # --------------------------------------------------------

    if n_fuera == 0:

        estado = (
            "### ✅ Datos dentro del dominio observado\n"
            "Los valores introducidos se encuentran dentro de "
            "los rangos observados durante el entrenamiento "
            "2017–2022."
        )

    else:

        nombres_fuera = [
            x["variable"]
            for x in resultado["variables_fuera_rango"]
        ]

        nombres_texto = ", ".join(
            nombres_fuera[:8]
        )

        if len(nombres_fuera) > 8:
            nombres_texto += ", ..."

        estado = (
            "### ⚠️ Posible extrapolación\n"
            f"Se han detectado **{n_fuera} variables** fuera del "
            "rango observado durante el entrenamiento.\n\n"
            f"Variables: `{nombres_texto}`"
        )

    # --------------------------------------------------------
    # Resumen principal
    # --------------------------------------------------------

    resumen = (
        f"## 🌾 Rendimiento estimado\n\n"
        f"# {pred_ensemble:,.0f} kg/ha\n\n"
        f"### {pred_t_ha:.2f} t/ha\n\n"
        f"**Cultivo:** "
        f"{'Cebada' if cultivo == 'barley' else 'Trigo blando'}  \n"
        f"**Provincia de referencia:** {provincia}"
    )

    aviso = (
        "### ℹ️ Alcance de la herramienta\n"
        "Este resultado corresponde a un **prototipo experimental**. "
        "El modelo fue entrenado con datos agregados a escala provincial "
        "de España y no ha sido validado como modelo operacional de "
        "predicción a escala de parcela."
    )

    return (
        resumen,
        pred_rf,
        pred_xgb,
        estado,
        aviso
    )

# ------------------------------------------------------------
# 5. CONSTRUIR INTERFAZ
# ------------------------------------------------------------

with gr.Blocks(
    title="Predicción de rendimiento agrícola"
) as demo:

    gr.Markdown(
        """
# 🌾 Predicción experimental de rendimiento agrícola

### Trigo blando y cebada

Herramienta demostrativa basada en un modelo de **fusión de datos
meteorológicos, edáficos, satelitales y territoriales**.

Selecciona una provincia y un cultivo para cargar valores históricos
de referencia. Después puedes sustituirlos por los datos de tu finca
o campaña antes de calcular la predicción.
        """
    )

    # --------------------------------------------------------
    # BLOQUE 1 - LOCALIZACIÓN Y CULTIVO
    # --------------------------------------------------------

    gr.Markdown("## 1. Localización y cultivo")

    with gr.Row():

        provincia_input = gr.Dropdown(
            choices=provincias_demo,
            value=provincias_demo[0],
            label="Provincia"
        )

        cultivo_input = gr.Dropdown(
            choices=[
                ("Cebada", "barley"),
                ("Trigo blando", "common_soft_wheat")
            ],
            value="barley",
            label="Cultivo"
        )

    boton_referencias = gr.Button(
        "📍 Cargar valores de referencia"
    )

    gr.Markdown(
        """
Los valores de referencia corresponden a medianas históricas
del periodo de entrenamiento 2017–2022 para la combinación
provincia + cultivo seleccionada.
        """
    )

    # --------------------------------------------------------
    # CREAR CAMPOS NUMÉRICOS
    # --------------------------------------------------------

    componentes_ui = {}

    # --------------------------------------------------------
    # BLOQUE 2 - METEOROLOGÍA
    # --------------------------------------------------------

    with gr.Accordion(
        "🌦 2. Datos meteorológicos",
        open=True
    ):

        gr.Markdown(
            """
Introduce o revisa las condiciones meteorológicas de la campaña.
Los valores mostrados inicialmente son referencias históricas.
            """
        )

        for variable in variables_era5:

            componente = gr.Number(
                value=float(medianas_train[variable]),
                label=etiqueta_variable(variable)
            )

            componentes_ui[variable] = componente

    # --------------------------------------------------------
    # BLOQUE 3 - SUELO
    # --------------------------------------------------------

    with gr.Accordion(
        "🌱 3. Propiedades del suelo",
        open=True
    ):

        for variable in variables_suelo:

            componente = gr.Number(
                value=float(medianas_train[variable]),
                label=etiqueta_variable(variable)
            )

            componentes_ui[variable] = componente

    # --------------------------------------------------------
    # BLOQUE 4 - ÍNDICES ESPECTRALES
    # --------------------------------------------------------

    with gr.Accordion(
        "🛰️ 4. Estado del cultivo / índices espectrales",
        open=True
    ):

        gr.Markdown(
            """
Los índices NDVI, EVI y NDMI representan información
relacionada con el vigor y el estado hídrico de la vegetación.
            """
        )

        for variable in variables_sentinel:

            componente = gr.Number(
                value=float(medianas_train[variable]),
                label=etiqueta_variable(variable)
            )

            componentes_ui[variable] = componente

    # --------------------------------------------------------
    # BLOQUE 5 - EUROCROPS
    # --------------------------------------------------------

    with gr.Accordion(
        "⚙️ 5. Información territorial avanzada",
        open=False
    ):

        gr.Markdown(
            """
Estas variables forman parte del modelo original.
Para un usuario no técnico pueden mantenerse con los valores
de referencia cargados automáticamente.
            """
        )

        for variable in variables_eurocrops:

            componente = gr.Number(
                value=float(medianas_train[variable]),
                label=etiqueta_variable(variable)
            )

            componentes_ui[variable] = componente

    # --------------------------------------------------------
    # BOTÓN PRINCIPAL
    # --------------------------------------------------------

    gr.Markdown("## 6. Predicción")

    boton_predecir = gr.Button(
        "🌾 CALCULAR RENDIMIENTO",
        variant="primary"
    )

    # --------------------------------------------------------
    # RESULTADOS
    # --------------------------------------------------------

    resultado_principal = gr.Markdown()

    with gr.Accordion(
        "Ver predicciones de los modelos individuales",
        open=False
    ):

        with gr.Row():

            resultado_rf = gr.Number(
                label="Random Forest (kg/ha)",
                interactive=False
            )

            resultado_xgb = gr.Number(
                label="XGBoost (kg/ha)",
                interactive=False
            )

    estado_dominio = gr.Markdown()
    aviso_metodologico = gr.Markdown()

    # --------------------------------------------------------
    # EVENTO: CARGAR REFERENCIAS
    # --------------------------------------------------------

    campos_ordenados = [
        componentes_ui[var]
        for var in variables_ui
    ]

    boton_referencias.click(
        fn=cargar_referencias_ui,
        inputs=[
            provincia_input,
            cultivo_input
        ],
        outputs=campos_ordenados
    )

    # --------------------------------------------------------
    # EVENTO: PREDECIR
    # --------------------------------------------------------

    boton_predecir.click(
        fn=predecir_desde_ui,
        inputs=[
            cultivo_input,
            provincia_input,
            *campos_ordenados
        ],
        outputs=[
            resultado_principal,
            resultado_rf,
            resultado_xgb,
            estado_dominio,
            aviso_metodologico
        ]
    )

print("Interfaz construida correctamente.")
print("PASO 14 FINALIZADO")

In [ ]:
# ============================================================
# PASO 15 - LANZAR LA DEMO EN GRADIO
# ============================================================

demo.launch(
    share=True,
    debug=True
)

In [ ]:
# ============================================================
# PASO 16 - LÓGICA PARA LA UI SIMPLIFICADA
# ============================================================

# Nombres amigables de los cultivos
CULTIVOS_UI = {
    "Cebada": "barley",
    "Trigo blando": "common_soft_wheat"
}

# Variables que el usuario podrá modificar de forma avanzada
# El resto seguirán siendo completadas automáticamente
# con los valores de referencia provincia + cultivo.

VARS_SUELO_UI = [
    "clay_pct_0_30cm",
    "sand_pct_0_30cm",
    "soc_g_kg_0_30cm",
    "ph_0_30cm",
    "bdod_g_cm3_0_30cm"
]

VARS_METEO_UI = [
    "temp_media_c_mar_abr",
    "temp_media_c_may_jun",
    "precipitacion_mm_mar_abr",
    "precipitacion_mm_may_jun",
    "precipitacion_mm_campana",
    "radiacion_mj_m2_mar_abr",
    "radiacion_mj_m2_may_jun",
    "radiacion_mj_m2_campana"
]

VARS_VEGETACION_UI = [
    "NDVI_may_jun",
    "NDVI_jul",
    "EVI_may_jun",
    "EVI_jul",
    "NDMI_may_jun",
    "NDMI_jul"
]


def obtener_referencia(provincia, cultivo_ui):
    """
    Recupera los valores históricos de referencia 2017-2022
    para una combinación provincia + cultivo.
    """

    cultivo_modelo = CULTIVOS_UI[cultivo_ui]

    ref = referencias_provincia_cultivo[
        (referencias_provincia_cultivo["provincia_estandar"] == provincia) &
        (referencias_provincia_cultivo["cultivo"] == cultivo_modelo)
    ]

    if len(ref) == 0:
        raise ValueError(
            f"No existen valores de referencia para "
            f"{provincia} - {cultivo_ui}"
        )

    return ref.iloc[0].copy()


def preparar_entrada_simplificada(
    provincia,
    cultivo_ui,
    superficie_parcela_ha
):
    """
    Prepara automáticamente los 48 predictores que necesita
    el modelo a partir de los valores históricos de referencia.

    La superficie de parcela se conserva como información de la
    simulación y se utilizará para calcular la producción total.

    IMPORTANTE:
    No sustituimos automáticamente superficie_eurocrops_ha ni
    superficie_media_recinto_ha por la superficie de la finca,
    porque no representan exactamente el mismo concepto.
    """

    ref = obtener_referencia(
        provincia,
        cultivo_ui
    )

    cultivo_modelo = CULTIVOS_UI[cultivo_ui]

    # Crear entrada con todos los predictores
    entrada = {}

    for variable in feature_cols:

        if variable == "cultivo":
            entrada[variable] = cultivo_modelo

        else:
            entrada[variable] = float(ref[variable])

    return entrada


def resumen_referencias(provincia, cultivo_ui):
    """
    Devuelve algunos valores principales para comprobar
    qué información territorial está utilizando la aplicación.
    """

    ref = obtener_referencia(
        provincia,
        cultivo_ui
    )

    resumen = {
        "Provincia": provincia,
        "Cultivo": cultivo_ui,
        "Arena suelo (%)": round(float(ref["sand_pct_0_30cm"]), 2),
        "Arcilla suelo (%)": round(float(ref["clay_pct_0_30cm"]), 2),
        "pH": round(float(ref["ph_0_30cm"]), 2),
        "Precipitación campaña (mm)": round(
            float(ref["precipitacion_mm_campana"]), 1
        ),
        "Temperatura media campaña (°C)": round(
            float(ref["temp_media_c_campana"]), 1
        ),
        "Radiación campaña (MJ/m²)": round(
            float(ref["radiacion_mj_m2_campana"]), 1
        )
    }

    return resumen


print("=" * 80)
print("PASO 16 - LÓGICA PARA UI SIMPLIFICADA")
print("=" * 80)

print("\nCultivos disponibles:")
print(list(CULTIVOS_UI.keys()))

print("\nProvincias disponibles:")
print(len(provincias_demo))

print("\nVariables avanzadas seleccionadas:")
print("Suelo:", len(VARS_SUELO_UI))
print("Meteorología:", len(VARS_METEO_UI))
print("Vegetación:", len(VARS_VEGETACION_UI))

# Prueba interna mínima
provincia_prueba = provincias_demo[0]
cultivo_prueba = "Cebada"

entrada_prueba = preparar_entrada_simplificada(
    provincia_prueba,
    cultivo_prueba,
    10.0
)

print("\nPredictores generados:", len(entrada_prueba))

assert len(entrada_prueba) == len(feature_cols)
assert set(entrada_prueba.keys()) == set(feature_cols)

print("\nLa entrada simplificada genera correctamente los 48 predictores.")
print("PASO 16 FINALIZADO")

In [ ]:
# ============================================================
# PASO 16 - LÓGICA PARA LA UI SIMPLIFICADA
# ============================================================

# Nombres amigables de los cultivos
CULTIVOS_UI = {
    "Cebada": "barley",
    "Trigo blando": "common_soft_wheat"
}

# Variables opcionales que el usuario podrá modificar
VARS_SUELO_UI = [
    "clay_pct_0_30cm",
    "sand_pct_0_30cm",
    "soc_g_kg_0_30cm",
    "ph_0_30cm",
    "bdod_g_cm3_0_30cm"
]

VARS_METEO_UI = [
    "temp_media_c_mar_abr",
    "temp_media_c_may_jun",
    "precipitacion_mm_mar_abr",
    "precipitacion_mm_may_jun",
    "precipitacion_mm_campana",
    "radiacion_mj_m2_mar_abr",
    "radiacion_mj_m2_may_jun",
    "radiacion_mj_m2_campana"
]

VARS_VEGETACION_UI = [
    "NDVI_may_jun",
    "NDVI_jul",
    "EVI_may_jun",
    "EVI_jul",
    "NDMI_may_jun",
    "NDMI_jul"
]


def obtener_referencia(provincia, cultivo_ui):
    """
    Recupera los valores históricos de referencia 2017-2022
    para una combinación provincia + cultivo.
    """

    cultivo_modelo = CULTIVOS_UI[cultivo_ui]

    ref = referencias_provincia_cultivo[
        (referencias_provincia_cultivo["provincia_estandar"] == provincia) &
        (referencias_provincia_cultivo["cultivo"] == cultivo_modelo)
    ]

    if len(ref) == 0:
        raise ValueError(
            f"No existen valores de referencia para "
            f"{provincia} - {cultivo_ui}"
        )

    return ref.iloc[0].copy()


def preparar_entrada_simplificada(
    provincia,
    cultivo_ui,
    superficie_parcela_ha
):
    """
    Prepara automáticamente los 48 predictores que necesita
    el modelo a partir de valores históricos de referencia.
    """

    ref = obtener_referencia(
        provincia,
        cultivo_ui
    )

    cultivo_modelo = CULTIVOS_UI[cultivo_ui]

    entrada = {}

    # IMPORTANTE: aquí usamos 'features', que es la lista
    # correcta definida en el Paso 4
    for variable in features:

        if variable == "cultivo":
            entrada[variable] = cultivo_modelo

        else:
            entrada[variable] = float(ref[variable])

    return entrada


def resumen_referencias(provincia, cultivo_ui):
    """
    Devuelve algunos valores principales de referencia.
    """

    ref = obtener_referencia(
        provincia,
        cultivo_ui
    )

    resumen = {
        "Provincia": provincia,
        "Cultivo": cultivo_ui,
        "Arena suelo (%)": round(float(ref["sand_pct_0_30cm"]), 2),
        "Arcilla suelo (%)": round(float(ref["clay_pct_0_30cm"]), 2),
        "pH": round(float(ref["ph_0_30cm"]), 2),
        "Precipitación campaña (mm)": round(
            float(ref["precipitacion_mm_campana"]), 1
        ),
        "Temperatura media campaña (°C)": round(
            float(ref["temp_media_c_campana"]), 1
        ),
        "Radiación campaña (MJ/m²)": round(
            float(ref["radiacion_mj_m2_campana"]), 1
        )
    }

    return resumen


print("=" * 80)
print("PASO 16 - LÓGICA PARA UI SIMPLIFICADA")
print("=" * 80)

print("\nCultivos disponibles:")
print(list(CULTIVOS_UI.keys()))

print("\nProvincias disponibles:")
print(len(provincias_demo))

print("\nVariables avanzadas seleccionadas:")
print("Suelo:", len(VARS_SUELO_UI))
print("Meteorología:", len(VARS_METEO_UI))
print("Vegetación:", len(VARS_VEGETACION_UI))

# Prueba mínima
provincia_prueba = provincias_demo[0]
cultivo_prueba = "Cebada"

entrada_prueba = preparar_entrada_simplificada(
    provincia_prueba,
    cultivo_prueba,
    10.0
)

print("\nPredictores generados:", len(entrada_prueba))

assert len(entrada_prueba) == len(features)
assert set(entrada_prueba.keys()) == set(features)

print("\nLa entrada simplificada genera correctamente los 48 predictores.")
print("PASO 16 FINALIZADO")

In [ ]:
# ============================================================
# PASO 17 - FUNCIÓN DE PREDICCIÓN SIMPLIFICADA
# ============================================================

def predecir_demo_simplificada(
    cultivo_ui,
    provincia,
    superficie_ha
):
    """
    Predicción simplificada para la demo.

    El usuario introduce:
    - cultivo
    - provincia
    - superficie de la parcela

    Los demás predictores se completan con las medianas
    históricas 2017-2022 de provincia + cultivo.
    """

    # ----------------------------------------
    # 1. Validar superficie
    # ----------------------------------------

    superficie_ha = float(superficie_ha)

    if superficie_ha <= 0:
        raise ValueError(
            "La superficie de la parcela debe ser mayor que 0 ha."
        )

    # ----------------------------------------
    # 2. Preparar los 48 predictores
    # ----------------------------------------

    entrada = preparar_entrada_simplificada(
        provincia=provincia,
        cultivo_ui=cultivo_ui,
        superficie_parcela_ha=superficie_ha
    )

    X_usuario = pd.DataFrame(
        [entrada],
        columns=features
    )

    # ----------------------------------------
    # 3. Predicciones
    # ----------------------------------------

    pred_rf = float(
        modelo_rf.predict(X_usuario)[0]
    )

    pred_xgb = float(
        modelo_xgb.predict(X_usuario)[0]
    )

    # Ensemble 50/50
    pred_ensemble = (
        0.50 * pred_rf +
        0.50 * pred_xgb
    )

    # Evitar resultados físicamente imposibles
    pred_ensemble = max(0.0, pred_ensemble)

    # ----------------------------------------
    # 4. Conversión de unidades
    # ----------------------------------------

    rendimiento_t_ha = pred_ensemble / 1000

    produccion_total_t = (
        pred_ensemble *
        superficie_ha
    ) / 1000

    # ----------------------------------------
    # 5. Información de referencia
    # ----------------------------------------

    referencias = resumen_referencias(
        provincia,
        cultivo_ui
    )

    # ----------------------------------------
    # 6. Resultado
    # ----------------------------------------

    resultado = {
        "cultivo": cultivo_ui,
        "provincia": provincia,
        "superficie_ha": superficie_ha,

        "rendimiento_kg_ha": pred_ensemble,
        "rendimiento_t_ha": rendimiento_t_ha,
        "produccion_total_t": produccion_total_t,

        "pred_RF_kg_ha": pred_rf,
        "pred_XGB_kg_ha": pred_xgb,

        "referencias": referencias
    }

    return resultado


# ============================================================
# PRUEBA
# ============================================================

resultado_prueba = predecir_demo_simplificada(
    cultivo_ui="Cebada",
    provincia=provincias_demo[0],
    superficie_ha=10
)

print("=" * 80)
print("PASO 17 - PRUEBA DE PREDICCIÓN SIMPLIFICADA")
print("=" * 80)

print("\nCultivo:")
print(resultado_prueba["cultivo"])

print("\nProvincia:")
print(resultado_prueba["provincia"])

print("\nSuperficie:")
print(
    f'{resultado_prueba["superficie_ha"]:.2f} ha'
)

print("\nRandom Forest:")
print(
    f'{resultado_prueba["pred_RF_kg_ha"]:.1f} kg/ha'
)

print("\nXGBoost:")
print(
    f'{resultado_prueba["pred_XGB_kg_ha"]:.1f} kg/ha'
)

print("\nEnsemble RF + XGB:")
print(
    f'{resultado_prueba["rendimiento_kg_ha"]:.1f} kg/ha'
)

print("\nRendimiento:")
print(
    f'{resultado_prueba["rendimiento_t_ha"]:.2f} t/ha'
)

print("\nProducción total estimada:")
print(
    f'{resultado_prueba["produccion_total_t"]:.2f} t'
)

print("\nPASO 17 FINALIZADO")

In [ ]:
# ============================================================
# PASO 18 - CONSTRUIR UI SIMPLIFICADA
# ============================================================

import gradio as gr


# ------------------------------------------------------------
# 1. FUNCIÓN PARA FORMATO NUMÉRICO ESPAÑOL
# ------------------------------------------------------------

def formato_es(numero, decimales=0):
    """
    Convierte:
    2907.45 -> 2.907,45
    """

    texto = f"{numero:,.{decimales}f}"

    texto = (
        texto
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )

    return texto


# ------------------------------------------------------------
# 2. FUNCIÓN QUE DEVUELVE EL RESULTADO A LA UI
# ------------------------------------------------------------

def ejecutar_demo(
    cultivo_ui,
    provincia,
    superficie_ha
):

    resultado = predecir_demo_simplificada(
        cultivo_ui=cultivo_ui,
        provincia=provincia,
        superficie_ha=superficie_ha
    )

    rendimiento_kg = resultado["rendimiento_kg_ha"]
    rendimiento_t = resultado["rendimiento_t_ha"]
    produccion_t = resultado["produccion_total_t"]

    rf = resultado["pred_RF_kg_ha"]
    xgb = resultado["pred_XGB_kg_ha"]

    refs = resultado["referencias"]

    # --------------------------------------------------------
    # RESULTADO PRINCIPAL
    # --------------------------------------------------------

    resultado_md = f"""
## 🌾 Rendimiento estimado

# {formato_es(rendimiento_kg, 0)} kg/ha

### {formato_es(rendimiento_t, 2)} t/ha

**Producción total estimada para {formato_es(float(superficie_ha), 2)} ha:**

## {formato_es(produccion_t, 2)} toneladas
"""

    # --------------------------------------------------------
    # CONTEXTO UTILIZADO
    # --------------------------------------------------------

    contexto_md = f"""
### 📍 Contexto utilizado

**Provincia:** {provincia}
**Cultivo:** {cultivo_ui}

La predicción se ha calculado utilizando valores históricos de
referencia del periodo **2017–2022** para esta combinación
provincia + cultivo.

Algunos valores de referencia utilizados:

- Temperatura media de campaña: **{formato_es(refs["Temperatura media campaña (°C)"], 1)} °C**
- Precipitación de campaña: **{formato_es(refs["Precipitación campaña (mm)"], 1)} mm**
- Radiación de campaña: **{formato_es(refs["Radiación campaña (MJ/m²)"], 1)} MJ/m²**
- Arena del suelo: **{formato_es(refs["Arena suelo (%)"], 1)} %**
- Arcilla del suelo: **{formato_es(refs["Arcilla suelo (%)"], 1)} %**
- pH: **{formato_es(refs["pH"], 2)}**
"""

    # --------------------------------------------------------
    # MODELOS INDIVIDUALES
    # --------------------------------------------------------

    modelos_md = f"""
### 🔬 Predicciones de los modelos

**Random Forest:** {formato_es(rf, 0)} kg/ha
**XGBoost:** {formato_es(xgb, 0)} kg/ha

**Predicción principal:** ensemble 50 % Random Forest + 50 % XGBoost.
"""

    return (
        resultado_md,
        contexto_md,
        modelos_md
    )


# ------------------------------------------------------------
# 3. CONSTRUIR INTERFAZ
# ------------------------------------------------------------

with gr.Blocks(
    title="Predicción de rendimiento agrícola"
) as demo_simplificada:

    gr.Markdown(
        """
# 🌾 Predicción de rendimiento agrícola

### Demo experimental para trigo blando y cebada

Selecciona el cultivo, la provincia donde se encuentra la finca
y su superficie.

La aplicación utiliza automáticamente información histórica de
referencia meteorológica, edáfica, satelital y territorial para
generar una estimación de rendimiento.

> **Prototipo demostrativo:** el modelo fue desarrollado con datos
> agregados a escala provincial y no constituye todavía un sistema
> validado de predicción a escala de parcela.
"""
    )

    # --------------------------------------------------------
    # DATOS DEL USUARIO
    # --------------------------------------------------------

    gr.Markdown("## 📍 Datos de la finca")

    with gr.Row():

        cultivo_input_v2 = gr.Dropdown(
            choices=[
                "Cebada",
                "Trigo blando"
            ],
            value="Cebada",
            label="Cultivo"
        )

        provincia_input_v2 = gr.Dropdown(
            choices=provincias_demo,
            value=provincias_demo[0],
            label="Provincia"
        )

        superficie_input_v2 = gr.Number(
            value=10,
            minimum=0.01,
            label="Superficie de la finca (ha)"
        )

    # --------------------------------------------------------
    # BOTÓN PRINCIPAL
    # --------------------------------------------------------

    boton_v2 = gr.Button(
        "🌾 CALCULAR RENDIMIENTO",
        variant="primary"
    )

    # --------------------------------------------------------
    # RESULTADO
    # --------------------------------------------------------

    resultado_v2 = gr.Markdown()

    # --------------------------------------------------------
    # INFORMACIÓN ADICIONAL
    # --------------------------------------------------------

    with gr.Accordion(
        "📊 Ver datos de referencia utilizados",
        open=False
    ):

        contexto_v2 = gr.Markdown()

    with gr.Accordion(
        "🔬 Ver modelos individuales",
        open=False
    ):

        modelos_v2 = gr.Markdown()

    # --------------------------------------------------------
    # AVISO FINAL
    # --------------------------------------------------------

    gr.Markdown(
        """
---

### ℹ️ Alcance de la demo

Esta herramienta muestra cómo podría utilizarse el modelo desarrollado
en el TFM dentro de una aplicación orientada a usuarios finales.

Para una herramienta operacional sería necesario incorporar más
campañas agrícolas, información a escala de parcela y conexiones
automáticas con fuentes meteorológicas, edáficas y satelitales.
"""
    )

    # --------------------------------------------------------
    # EVENTO
    # --------------------------------------------------------

    boton_v2.click(
        fn=ejecutar_demo,
        inputs=[
            cultivo_input_v2,
            provincia_input_v2,
            superficie_input_v2
        ],
        outputs=[
            resultado_v2,
            contexto_v2,
            modelos_v2
        ]
    )


print("Nueva interfaz simplificada construida correctamente.")
print("PASO 18 FINALIZADO")

In [ ]:
# ============================================================
# PASO 19 - LANZAR LA DEMO SIMPLIFICADA
# ============================================================

demo_simplificada.launch(
    share=True,
    debug=True
)

In [ ]:
# ============================================================
# PASO 20 - LÓGICA HÍBRIDA:
# DATOS DEL USUARIO + VALORES DE REFERENCIA
# ============================================================

# Variables opcionales que permitiremos modificar en la UI.
# Si el usuario no introduce un valor, se conserva
# automáticamente el valor histórico de referencia.

VARS_USUARIO_METEO = [
    "temp_media_c_mar_abr",
    "temp_media_c_may_jun",
    "precipitacion_mm_mar_abr",
    "precipitacion_mm_may_jun",
    "precipitacion_mm_campana",
    "radiacion_mj_m2_mar_abr",
    "radiacion_mj_m2_may_jun",
    "radiacion_mj_m2_campana"
]

VARS_USUARIO_SUELO = [
    "clay_pct_0_30cm",
    "sand_pct_0_30cm",
    "soc_g_kg_0_30cm",
    "ph_0_30cm",
    "bdod_g_cm3_0_30cm"
]

VARS_USUARIO_VEGETACION = [
    "NDVI_may_jun",
    "NDVI_jul",
    "EVI_may_jun",
    "EVI_jul",
    "NDMI_may_jun",
    "NDMI_jul"
]

VARS_USUARIO = (
    VARS_USUARIO_METEO
    + VARS_USUARIO_SUELO
    + VARS_USUARIO_VEGETACION
)


def preparar_entrada_hibrida(
    provincia,
    cultivo_ui,
    datos_usuario
):
    """
    Construye los 48 predictores del modelo.

    1. Parte de los valores históricos de referencia
       para provincia + cultivo.
    2. Sustituye únicamente las variables para las que
       el usuario haya introducido un valor.
    """

    ref = obtener_referencia(
        provincia,
        cultivo_ui
    )

    cultivo_modelo = CULTIVOS_UI[cultivo_ui]

    entrada = {}

    # Crear entrada base con referencias históricas
    for variable in features:

        if variable == "cultivo":
            entrada[variable] = cultivo_modelo
        else:
            entrada[variable] = float(ref[variable])

    # Sustituir únicamente datos aportados por el usuario
    variables_personalizadas = []

    for variable, valor in datos_usuario.items():

        if valor is not None:

            entrada[variable] = float(valor)
            variables_personalizadas.append(variable)

    return entrada, variables_personalizadas


def predecir_demo_hibrida(
    cultivo_ui,
    provincia,
    superficie_ha,
    datos_usuario
):
    """
    Predicción utilizando referencias históricas y,
    cuando existan, datos aportados por el usuario.
    """

    superficie_ha = float(superficie_ha)

    if superficie_ha <= 0:
        raise ValueError(
            "La superficie de la finca debe ser mayor que 0 ha."
        )

    # --------------------------------------------------------
    # Preparar predictores
    # --------------------------------------------------------

    entrada, variables_personalizadas = preparar_entrada_hibrida(
        provincia=provincia,
        cultivo_ui=cultivo_ui,
        datos_usuario=datos_usuario
    )

    X_usuario = pd.DataFrame(
        [entrada],
        columns=features
    )

    # --------------------------------------------------------
    # Predicciones
    # --------------------------------------------------------

    pred_rf = float(
        modelo_rf.predict(X_usuario)[0]
    )

    pred_xgb = float(
        modelo_xgb.predict(X_usuario)[0]
    )

    pred_ensemble = (
        0.50 * pred_rf
        + 0.50 * pred_xgb
    )

    pred_ensemble = max(
        0.0,
        pred_ensemble
    )

    rendimiento_t_ha = (
        pred_ensemble / 1000
    )

    produccion_total_t = (
        pred_ensemble
        * superficie_ha
    ) / 1000

    # --------------------------------------------------------
    # Comprobar dominio del entrenamiento
    # --------------------------------------------------------

    fuera_rango = []

    for variable in num_cols:

        valor = float(
            X_usuario.iloc[0][variable]
        )

        minimo = float(
            minimos_train[variable]
        )

        maximo = float(
            maximos_train[variable]
        )

        if valor < minimo or valor > maximo:

            fuera_rango.append(
                variable
            )

    return {
        "cultivo": cultivo_ui,
        "provincia": provincia,
        "superficie_ha": superficie_ha,

        "rendimiento_kg_ha": pred_ensemble,
        "rendimiento_t_ha": rendimiento_t_ha,
        "produccion_total_t": produccion_total_t,

        "pred_RF_kg_ha": pred_rf,
        "pred_XGB_kg_ha": pred_xgb,

        "variables_personalizadas":
            variables_personalizadas,

        "n_variables_personalizadas":
            len(variables_personalizadas),

        "variables_fuera_rango":
            fuera_rango,

        "n_variables_fuera_rango":
            len(fuera_rango)
    }


print("=" * 80)
print("PASO 20 - LÓGICA HÍBRIDA")
print("=" * 80)

print("\nVariables opcionales disponibles:")
print("Meteorología:", len(VARS_USUARIO_METEO))
print("Suelo:", len(VARS_USUARIO_SUELO))
print("Vegetación:", len(VARS_USUARIO_VEGETACION))

print("\nTotal variables opcionales:")
print(len(VARS_USUARIO))

print("\nSi el usuario deja un campo vacío:")
print("→ se utiliza automáticamente el valor de referencia.")

print("\nPASO 20 FINALIZADO")

In [ ]:
# ============================================================
# PASO 21 - CONSTRUIR UI HÍBRIDA CON MÁXIMA PERSONALIZACIÓN
# ============================================================

import gradio as gr


# ------------------------------------------------------------
# 1. VARIABLES PERSONALIZABLES
# ------------------------------------------------------------

# Resumen general de campaña: las más fáciles de entender
VARS_CLIMA_RESUMEN = [
    "temp_media_c_campana",
    "temp_rocio_c_campana",
    "precipitacion_mm_campana",
    "radiacion_mj_m2_campana"
]

# Variables meteorológicas detalladas por periodos
VARS_CLIMA_DETALLE = [
    variable
    for variable in variables_era5
    if variable not in VARS_CLIMA_RESUMEN
]

# Propiedades del suelo
VARS_SUELO_PERSONALIZABLES = variables_suelo.copy()

# Índices Sentinel-2
VARS_VEGETACION_PERSONALIZABLES = variables_sentinel.copy()

# Todas las variables que el stakeholder puede sustituir
VARS_PERSONALIZABLES = (
    VARS_CLIMA_RESUMEN
    + VARS_CLIMA_DETALLE
    + VARS_SUELO_PERSONALIZABLES
    + VARS_VEGETACION_PERSONALIZABLES
)

print("Variables personalizables:", len(VARS_PERSONALIZABLES))


# ------------------------------------------------------------
# 2. FUNCIÓN DE PREDICCIÓN PARA LA UI
# ------------------------------------------------------------

def ejecutar_demo_hibrida_ui(
    cultivo_ui,
    provincia,
    superficie_ha,
    *valores_usuario
):

    # Asociar cada campo de la UI con su variable real
    datos_usuario = dict(
        zip(
            VARS_PERSONALIZABLES,
            valores_usuario
        )
    )

    # Ejecutar modelo híbrido
    resultado = predecir_demo_hibrida(
        cultivo_ui=cultivo_ui,
        provincia=provincia,
        superficie_ha=superficie_ha,
        datos_usuario=datos_usuario
    )

    rendimiento_kg = resultado["rendimiento_kg_ha"]
    rendimiento_t = resultado["rendimiento_t_ha"]
    produccion_t = resultado["produccion_total_t"]

    rf = resultado["pred_RF_kg_ha"]
    xgb = resultado["pred_XGB_kg_ha"]

    n_personalizadas = resultado[
        "n_variables_personalizadas"
    ]

    n_referencia = (
        len(num_cols)
        - n_personalizadas
    )

    n_fuera = resultado[
        "n_variables_fuera_rango"
    ]

    # --------------------------------------------------------
    # RESULTADO PRINCIPAL
    # --------------------------------------------------------

    resultado_md = f"""
## 🌾 Rendimiento estimado

# {formato_es(rendimiento_kg, 0)} kg/ha

### {formato_es(rendimiento_t, 2)} t/ha

**Producción total estimada para {formato_es(float(superficie_ha), 2)} ha:**

## {formato_es(produccion_t, 2)} toneladas
"""

    # --------------------------------------------------------
    # INFORMACIÓN UTILIZADA
    # --------------------------------------------------------

    if n_personalizadas == 0:

        datos_md = f"""
### 📊 Información utilizada

No se han introducido datos adicionales.

La estimación se ha realizado utilizando los **valores históricos
de referencia 2017–2022** correspondientes a:

- **Provincia:** {provincia}
- **Cultivo:** {cultivo_ui}

Los 47 predictores numéricos requeridos por el modelo se han
completado automáticamente.
"""

    else:

        datos_md = f"""
### 📊 Información utilizada

**Datos personalizados introducidos:** {n_personalizadas}

**Variables completadas automáticamente con referencias
históricas:** {n_referencia}

Las variables sin información proporcionada por el usuario
se han completado utilizando los valores de referencia
2017–2022 para **{provincia} – {cultivo_ui}**.
"""

    # --------------------------------------------------------
    # CONTROL DE DOMINIO
    # --------------------------------------------------------

    if n_fuera == 0:

        dominio_md = """
### ✅ Valores compatibles con el dominio del modelo

Los datos utilizados se encuentran dentro de los rangos
observados en el conjunto de entrenamiento 2017–2022.
"""

    else:

        lista_fuera = resultado[
            "variables_fuera_rango"
        ]

        texto_fuera = ", ".join(
            lista_fuera[:10]
        )

        if len(lista_fuera) > 10:
            texto_fuera += ", ..."

        dominio_md = f"""
### ⚠️ Posible extrapolación

Se han detectado **{n_fuera} variables** fuera del rango
observado durante el entrenamiento 2017–2022.

Variables:

`{texto_fuera}`

La predicción debe interpretarse con mayor precaución.
"""

    # --------------------------------------------------------
    # MODELOS INDIVIDUALES
    # --------------------------------------------------------

    modelos_md = f"""
### 🔬 Modelos

**Random Forest:** {formato_es(rf, 0)} kg/ha
**XGBoost:** {formato_es(xgb, 0)} kg/ha

**Predicción principal:** ensemble 50 % Random Forest +
50 % XGBoost.
"""

    return (
        resultado_md,
        datos_md,
        dominio_md,
        modelos_md
    )


# ------------------------------------------------------------
# 3. CONSTRUIR INTERFAZ
# ------------------------------------------------------------

with gr.Blocks(
    title="Predicción de rendimiento agrícola"
) as demo_hibrida:

    gr.Markdown(
        """
# 🌾 Predicción de rendimiento agrícola

### Demo experimental para trigo blando y cebada

Para obtener una estimación básica solo necesitas indicar:

**Cultivo + Provincia + Superficie de la finca**

La aplicación completará automáticamente la información restante
con valores históricos de referencia del periodo **2017–2022**.

Si dispones de información propia sobre clima, suelo o estado del
cultivo, puedes introducirla en los apartados opcionales.
**Los campos que dejes vacíos se completarán automáticamente.**

> El modelo fue desarrollado con datos agregados a escala provincial.
> Esta aplicación es un prototipo demostrativo y no constituye todavía
> un sistema operacional validado a escala de parcela.
"""
    )

    # ========================================================
    # BLOQUE BÁSICO
    # ========================================================

    gr.Markdown("## 📍 Datos básicos de la finca")

    with gr.Row():

        cultivo_h = gr.Dropdown(
            choices=[
                "Cebada",
                "Trigo blando"
            ],
            value="Cebada",
            label="Cultivo"
        )

        provincia_h = gr.Dropdown(
            choices=provincias_demo,
            value=provincias_demo[0],
            label="Provincia"
        )

        superficie_h = gr.Number(
            value=10,
            minimum=0.01,
            label="Superficie de la finca (ha)"
        )

    gr.Markdown(
        """
### ¿Tienes más información?

Los siguientes apartados son **opcionales**.

Puedes introducir únicamente los datos que conozcas.
Deja vacío cualquier campo para utilizar el valor histórico
de referencia.
"""
    )

    componentes_hibridos = {}


    # ========================================================
    # METEOROLOGÍA SIMPLE
    # ========================================================

    with gr.Accordion(
        "🌦️ Datos meteorológicos generales — opcional",
        open=False
    ):

        gr.Markdown(
            """
Si dispones de información general de la campaña,
puedes introducirla aquí.

No es necesario completar todos los campos.
"""
        )

        for variable in VARS_CLIMA_RESUMEN:

            componente = gr.Number(
                value=None,
                label=etiqueta_variable(variable),
                placeholder="Usar valor histórico de referencia"
            )

            componentes_hibridos[
                variable
            ] = componente


    # ========================================================
    # METEOROLOGÍA DETALLADA
    # ========================================================

    with gr.Accordion(
        "🌦️ Meteorología detallada por periodos — avanzado",
        open=False
    ):

        gr.Markdown(
            """
Utiliza este apartado únicamente si dispones de datos
meteorológicos desagregados por periodos de la campaña.

Los campos vacíos seguirán utilizando referencias históricas.
"""
        )

        for variable in VARS_CLIMA_DETALLE:

            componente = gr.Number(
                value=None,
                label=etiqueta_variable(variable),
                placeholder="Usar referencia"
            )

            componentes_hibridos[
                variable
            ] = componente


    # ========================================================
    # SUELO
    # ========================================================

    with gr.Accordion(
        "🌱 Propiedades del suelo — opcional",
        open=False
    ):

        gr.Markdown(
            """
Si dispones de análisis de suelo puedes introducir aquí
los valores conocidos.

No es necesario disponer de todas las propiedades.
"""
        )

        for variable in VARS_SUELO_PERSONALIZABLES:

            componente = gr.Number(
                value=None,
                label=etiqueta_variable(variable),
                placeholder="Usar valor de referencia"
            )

            componentes_hibridos[
                variable
            ] = componente


    # ========================================================
    # VEGETACIÓN / SATÉLITE
    # ========================================================

    with gr.Accordion(
        "🛰️ Datos satelitales / estado del cultivo — avanzado",
        open=False
    ):

        gr.Markdown(
            """
El modelo utiliza índices NDVI, EVI y NDMI correspondientes
a diferentes momentos de la campaña.

Rellena estos campos únicamente si dispones de esta información.
Si no la tienes, la aplicación utilizará automáticamente
los valores históricos de referencia.
"""
        )

        for variable in VARS_VEGETACION_PERSONALIZABLES:

            componente = gr.Number(
                value=None,
                label=etiqueta_variable(variable),
                placeholder="Usar referencia"
            )

            componentes_hibridos[
                variable
            ] = componente


    # ========================================================
    # INFORMACIÓN AUTOMÁTICA
    # ========================================================

    with gr.Accordion(
        "🗺️ Información territorial utilizada automáticamente",
        open=False
    ):

        gr.Markdown(
            """
Las variables territoriales derivadas de **EuroCrops**
se incorporan automáticamente según la provincia y el cultivo.

No se solicitan al usuario porque representan información
agregada utilizada durante el entrenamiento del modelo y
no equivalen directamente a las características individuales
de una finca.
"""
        )


    # ========================================================
    # PREDICCIÓN
    # ========================================================

    gr.Markdown("## 🌾 Obtener estimación")

    boton_h = gr.Button(
        "CALCULAR RENDIMIENTO",
        variant="primary"
    )

    resultado_h = gr.Markdown()


    # ========================================================
    # INFORMACIÓN DEL RESULTADO
    # ========================================================

    with gr.Accordion(
        "📊 Ver cómo se ha construido la predicción",
        open=False
    ):

        datos_h = gr.Markdown()

    dominio_h = gr.Markdown()

    with gr.Accordion(
        "🔬 Ver modelos individuales",
        open=False
    ):

        modelos_h = gr.Markdown()


    # ========================================================
    # EVENTO
    # ========================================================

    campos_hibridos_ordenados = [
        componentes_hibridos[var]
        for var in VARS_PERSONALIZABLES
    ]

    boton_h.click(
        fn=ejecutar_demo_hibrida_ui,
        inputs=[
            cultivo_h,
            provincia_h,
            superficie_h,
            *campos_hibridos_ordenados
        ],
        outputs=[
            resultado_h,
            datos_h,
            dominio_h,
            modelos_h
        ]
    )


print("=" * 80)
print("PASO 21 - UI HÍBRIDA")
print("=" * 80)

print(
    "Variables que el usuario puede personalizar:",
    len(VARS_PERSONALIZABLES)
)

print(
    "Variables EuroCrops automatizadas:",
    len(variables_eurocrops)
)

print("\nInterfaz híbrida construida correctamente.")
print("PASO 21 FINALIZADO")

In [ ]:
# ============================================================
# PASO 22 - LANZAR LA DEMO HÍBRIDA
# ============================================================

demo_hibrida.launch(
    share=True,
    debug=True
)

In [ ]:
# ============================================================
# PASO 23 - MEJORAR INFORMACIÓN DE REFERENCIA Y SIMPLIFICAR TEXTOS
# ============================================================

def ejecutar_demo_hibrida_ui_v2(
    cultivo_ui,
    provincia,
    superficie_ha,
    *valores_usuario
):

    datos_usuario = dict(
        zip(
            VARS_PERSONALIZABLES,
            valores_usuario
        )
    )

    resultado = predecir_demo_hibrida(
        cultivo_ui=cultivo_ui,
        provincia=provincia,
        superficie_ha=superficie_ha,
        datos_usuario=datos_usuario
    )

    rendimiento_kg = resultado["rendimiento_kg_ha"]
    rendimiento_t = resultado["rendimiento_t_ha"]
    produccion_t = resultado["produccion_total_t"]

    rf = resultado["pred_RF_kg_ha"]
    xgb = resultado["pred_XGB_kg_ha"]

    n_personalizadas = resultado["n_variables_personalizadas"]
    n_referencia = len(num_cols) - n_personalizadas
    n_fuera = resultado["n_variables_fuera_rango"]

    # --------------------------------------------------------
    # Recuperar referencias utilizadas
    # --------------------------------------------------------

    ref = obtener_referencia(
        provincia,
        cultivo_ui
    )

    # --------------------------------------------------------
    # RESULTADO PRINCIPAL
    # --------------------------------------------------------

    resultado_md = f"""
## 🌾 Rendimiento estimado

# {formato_es(rendimiento_kg, 0)} kg/ha

### {formato_es(rendimiento_t, 2)} t/ha

**Producción total estimada para {formato_es(float(superficie_ha), 2)} ha:**

## {formato_es(produccion_t, 2)} toneladas
"""

    # --------------------------------------------------------
    # INFORMACIÓN UTILIZADA
    # --------------------------------------------------------

    if n_personalizadas == 0:

        datos_md = f"""
### 📊 Datos utilizados para la estimación

No se han introducido datos adicionales.

La aplicación ha utilizado valores históricos de referencia
para **{provincia} – {cultivo_ui}**, calculados a partir del periodo
**2017–2022**.

#### Valores principales de referencia utilizados

- Temperatura media de campaña: **{formato_es(float(ref["temp_media_c_campana"]), 1)} °C**
- Precipitación de campaña: **{formato_es(float(ref["precipitacion_mm_campana"]), 1)} mm**
- Radiación de campaña: **{formato_es(float(ref["radiacion_mj_m2_campana"]), 1)} MJ/m²**
- Arena del suelo: **{formato_es(float(ref["sand_pct_0_30cm"]), 1)} %**
- Arcilla del suelo: **{formato_es(float(ref["clay_pct_0_30cm"]), 1)} %**
- pH del suelo: **{formato_es(float(ref["ph_0_30cm"]), 2)}**
- NDVI mayo–junio: **{formato_es(float(ref["NDVI_may_jun"]), 3)}**
- NDMI mayo–junio: **{formato_es(float(ref["NDMI_may_jun"]), 3)}**

Los demás parámetros necesarios para el modelo también se han
completado automáticamente con sus valores históricos de referencia.
"""

    else:

        datos_md = f"""
### 📊 Datos utilizados para la estimación

**Datos introducidos por el usuario:** {n_personalizadas}

**Variables completadas automáticamente con valores de referencia:** {n_referencia}

Los campos que no has rellenado se han completado con valores históricos
de referencia para **{provincia} – {cultivo_ui}** del periodo **2017–2022**.

#### Principales valores de referencia utilizados

- Temperatura media de campaña: **{formato_es(float(ref["temp_media_c_campana"]), 1)} °C**
- Precipitación de campaña: **{formato_es(float(ref["precipitacion_mm_campana"]), 1)} mm**
- Radiación de campaña: **{formato_es(float(ref["radiacion_mj_m2_campana"]), 1)} MJ/m²**
- Arena del suelo: **{formato_es(float(ref["sand_pct_0_30cm"]), 1)} %**
- Arcilla del suelo: **{formato_es(float(ref["clay_pct_0_30cm"]), 1)} %**
- pH del suelo: **{formato_es(float(ref["ph_0_30cm"]), 2)}**
- NDVI mayo–junio: **{formato_es(float(ref["NDVI_may_jun"]), 3)}**
- NDMI mayo–junio: **{formato_es(float(ref["NDMI_may_jun"]), 3)}**
"""

    # --------------------------------------------------------
    # CONTROL DE DOMINIO
    # --------------------------------------------------------

    if n_fuera == 0:

        dominio_md = """
### ✅ Datos dentro del rango habitual del modelo

Los valores utilizados se encuentran dentro de los rangos
observados en los datos de entrenamiento.
"""

    else:

        lista_fuera = resultado["variables_fuera_rango"]

        texto_fuera = ", ".join(
            lista_fuera[:10]
        )

        if len(lista_fuera) > 10:
            texto_fuera += ", ..."

        dominio_md = f"""
### ⚠️ Algunos valores son poco habituales

Se han detectado **{n_fuera} variables** fuera del rango observado
en los datos de entrenamiento.

`{texto_fuera}`

La estimación debe interpretarse con mayor precaución.
"""

    # --------------------------------------------------------
    # MODELOS INDIVIDUALES
    # --------------------------------------------------------

    modelos_md = f"""
### 🔬 Información del modelo

**Random Forest:** {formato_es(rf, 0)} kg/ha
**XGBoost:** {formato_es(xgb, 0)} kg/ha

La estimación principal combina ambos modelos al 50 %.
"""

    return (
        resultado_md,
        datos_md,
        dominio_md,
        modelos_md
    )


print("PASO 23 - FUNCIÓN DE SALIDA ACTUALIZADA")
print("PASO 23 FINALIZADO")

In [ ]:
# ============================================================
# PASO 23B - ACTUALIZAR TEXTOS Y CONEXIÓN DE LA UI
# ============================================================

# Cambiar el contenido del apartado territorial
# construyendo una nueva versión de la interfaz en el siguiente paso.
# De momento solo dejamos preparada la nueva función.

print("Nueva función preparada para la versión final de la interfaz.")
print("PASO 23B FINALIZADO")

In [ ]:
# ============================================================
# PASO 24 - CREAR VERSIÓN FINAL DE LA INTERFAZ HÍBRIDA
# CON REDACCIÓN PROFESIONAL
# ============================================================

import gradio as gr

with gr.Blocks(
    title="Predicción de rendimiento agrícola"
) as demo_final:

    gr.Markdown(
        """
# 🌾 Predicción de rendimiento agrícola

### Demo experimental para trigo blando y cebada

Para obtener una estimación básica solo es necesario indicar:

**Cultivo + Provincia + Superficie de la finca**

La aplicación completará automáticamente la información restante
con valores históricos de referencia del periodo **2017–2022**.

Si dispone de información propia sobre clima, suelo o estado del
cultivo, puede introducirla en los apartados opcionales.

**Los campos que queden vacíos se completarán automáticamente.**

> Esta herramienta es un prototipo demostrativo desarrollado a partir
> del modelo del TFM. El modelo fue entrenado con datos agregados
> a escala provincial.
"""
    )

    # ========================================================
    # BLOQUE BÁSICO
    # ========================================================

    gr.Markdown("## 📍 Datos básicos de la finca")

    with gr.Row():

        cultivo_final = gr.Dropdown(
            choices=[
                "Cebada",
                "Trigo blando"
            ],
            value="Cebada",
            label="Cultivo"
        )

        provincia_final = gr.Dropdown(
            choices=provincias_demo,
            value=provincias_demo[0],
            label="Provincia"
        )

        superficie_final = gr.Number(
            value=10,
            minimum=0.01,
            label="Superficie de la finca (ha)"
        )

    gr.Markdown(
        """
### ¿Dispone de información adicional?

Los siguientes apartados son **opcionales**.

Introduzca únicamente los datos disponibles.
Los campos que queden vacíos se completarán automáticamente
con valores históricos de referencia.
"""
    )

    componentes_finales = {}

    # ========================================================
    # CLIMA GENERAL
    # ========================================================

    with gr.Accordion(
        "🌦️ Datos meteorológicos generales — opcional",
        open=False
    ):

        gr.Markdown(
            """
Introduzca los datos generales de la campaña si dispone de ellos.

No es necesario completar todos los campos.
Los campos vacíos se completarán automáticamente con valores
históricos de referencia.
"""
        )

        for variable in VARS_CLIMA_RESUMEN:

            componente = gr.Number(
                value=None,
                label=etiqueta_variable(variable),
                placeholder="Usar valor histórico de referencia"
            )

            componentes_finales[variable] = componente

    # ========================================================
    # CLIMA DETALLADO
    # ========================================================

    with gr.Accordion(
        "🌦️ Meteorología detallada por periodos — avanzado",
        open=False
    ):

        gr.Markdown(
            """
Utilice este apartado únicamente si dispone de datos meteorológicos
más detallados para diferentes periodos de la campaña.

Los campos vacíos se completarán automáticamente con valores
históricos de referencia.
"""
        )

        for variable in VARS_CLIMA_DETALLE:

            componente = gr.Number(
                value=None,
                label=etiqueta_variable(variable),
                placeholder="Usar valor histórico de referencia"
            )

            componentes_finales[variable] = componente

    # ========================================================
    # SUELO
    # ========================================================

    with gr.Accordion(
        "🌱 Propiedades del suelo — opcional",
        open=False
    ):

        gr.Markdown(
            """
Si dispone de un análisis de suelo, introduzca los valores disponibles.

No es necesario completar todas las propiedades.
Los campos vacíos se completarán automáticamente con valores
históricos de referencia.
"""
        )

        for variable in VARS_SUELO_PERSONALIZABLES:

            componente = gr.Number(
                value=None,
                label=etiqueta_variable(variable),
                placeholder="Usar valor histórico de referencia"
            )

            componentes_finales[variable] = componente

    # ========================================================
    # VEGETACIÓN / SATÉLITE
    # ========================================================

    with gr.Accordion(
        "🛰️ Datos satelitales / estado del cultivo — avanzado",
        open=False
    ):

        gr.Markdown(
            """
Introduzca estos valores únicamente si dispone de información
satelital o de índices de vegetación.

En caso contrario, deje los campos vacíos y la aplicación utilizará
automáticamente valores históricos de referencia.
"""
        )

        for variable in VARS_VEGETACION_PERSONALIZABLES:

            componente = gr.Number(
                value=None,
                label=etiqueta_variable(variable),
                placeholder="Usar valor histórico de referencia"
            )

            componentes_finales[variable] = componente

    # ========================================================
    # INFORMACIÓN TERRITORIAL
    # ========================================================

    with gr.Accordion(
        "🗺️ Información territorial de referencia",
        open=False
    ):

        gr.Markdown(
            """
La aplicación incorpora automáticamente información territorial
asociada a la provincia y al cultivo seleccionados.
"""
        )

    # ========================================================
    # BOTÓN DE PREDICCIÓN
    # ========================================================

    gr.Markdown("## 🌾 Obtener estimación")

    boton_final = gr.Button(
        "CALCULAR RENDIMIENTO",
        variant="primary"
    )

    # ========================================================
    # RESULTADOS
    # ========================================================

    resultado_final = gr.Markdown()

    with gr.Accordion(
        "📊 Ver datos utilizados para la estimación",
        open=False
    ):

        datos_final = gr.Markdown()

    dominio_final = gr.Markdown()

    with gr.Accordion(
        "🔬 Ver información del modelo",
        open=False
    ):

        modelos_final = gr.Markdown()

    # ========================================================
    # AVISO FINAL
    # ========================================================

    gr.Markdown(
        """
---

### ℹ️ Alcance de la herramienta

Esta demo muestra cómo puede integrarse el modelo desarrollado
en el TFM dentro de una herramienta sencilla orientada a usuarios finales.

Para avanzar hacia una aplicación operacional sería necesario incorporar
más campañas, datos a escala de parcela y conexiones automáticas con
fuentes meteorológicas, edáficas y satelitales.
"""
    )

    # ========================================================
    # CONEXIÓN DEL BOTÓN
    # ========================================================

    campos_finales_ordenados = [
        componentes_finales[var]
        for var in VARS_PERSONALIZABLES
    ]

    boton_final.click(
        fn=ejecutar_demo_hibrida_ui_v2,
        inputs=[
            cultivo_final,
            provincia_final,
            superficie_final,
            *campos_finales_ordenados
        ],
        outputs=[
            resultado_final,
            datos_final,
            dominio_final,
            modelos_final
        ]
    )


print("=" * 80)
print("PASO 24 - INTERFAZ FINAL")
print("=" * 80)

print(
    "Variables personalizables:",
    len(VARS_PERSONALIZABLES)
)

print(
    "Variables completadas automáticamente si faltan:",
    len(num_cols)
)

print("\nInterfaz final construida correctamente.")
print("PASO 24 FINALIZADO")

In [ ]:
# ============================================================
# PASO 25 - LANZAR LA INTERFAZ FINAL
# ============================================================

demo_final.launch(
    share=True,
    debug=True
)

# Despliegue en la web Streamlit Community Cloud (desestimo HuggingFace_Deploy porque es de pago)

In [ ]:
# ============================================================
# PASO 26 - CREAR CARPETA PARA DESPLIEGUE EN HuggingFace_Deploy
# ============================================================

import os

# Carpeta específica para la versión web
DEPLOY_DIR = (
    "/content/TFM_Rendimiento_cultivos/"
    "TFM_Rendimiento_cultivos/"
    "06_Demo_UI/"
    "HuggingFace_Deploy"
)

# Crear carpeta
os.makedirs(
    DEPLOY_DIR,
    exist_ok=True
)

print("=" * 80)
print("PASO 26 - CARPETA DE DESPLIEGUE WEB")
print("=" * 80)

print("\nCarpeta:")
print(DEPLOY_DIR)

print("\nExiste:")
print(os.path.exists(DEPLOY_DIR))

print("\nPASO 26 FINALIZADO")

In [ ]:
# ============================================================
# PASO 27 - CREAR CARPETA PARA DESPLIEGUE GRATUITO EN STREAMLIT
# ============================================================

import os

STREAMLIT_DIR = (
    "/content/TFM_Rendimiento_cultivos/"
    "TFM_Rendimiento_cultivos/"
    "06_Demo_UI/"
    "Streamlit_Deploy"
)

os.makedirs(
    STREAMLIT_DIR,
    exist_ok=True
)

print("=" * 80)
print("PASO 27 - CARPETA STREAMLIT")
print("=" * 80)

print("\nCarpeta:")
print(STREAMLIT_DIR)

print("\nExiste:")
print(os.path.exists(STREAMLIT_DIR))

print("\nPASO 27 FINALIZADO")

In [ ]:
# ============================================================
# PASO 28 - COPIAR ARCHIVOS NECESARIOS PARA STREAMLIT
# ============================================================

import os
import shutil

archivos_origen = {
    "RF_tuned_TFM.joblib":
        f"{MODELOS_DIR}/RF_tuned_TFM.joblib",

    "XGB_tuned_TFM.joblib":
        f"{MODELOS_DIR}/XGB_tuned_TFM.joblib",

    "referencias_provincia_cultivo_2017_2022.csv":
        f"{CONFIG_DIR}/referencias_provincia_cultivo_2017_2022.csv",

    "rangos_entrenamiento_2017_2022.csv":
        f"{CONFIG_DIR}/rangos_entrenamiento_2017_2022.csv"
}

for nombre_destino, origen in archivos_origen.items():

    destino = os.path.join(
        STREAMLIT_DIR,
        nombre_destino
    )

    shutil.copy2(
        origen,
        destino
    )

print("=" * 80)
print("PASO 28 - ARCHIVOS PARA STREAMLIT")
print("=" * 80)

print("\nArchivos preparados:")

for nombre in archivos_origen:
    print(
        f"- {nombre} -> "
        f"{os.path.exists(os.path.join(STREAMLIT_DIR, nombre))}"
    )

print("\nPASO 28 FINALIZADO")

In [ ]:
# ============================================================
# PASO 29 - CREAR app.py PARA STREAMLIT
# ============================================================

import os

APP_PATH = os.path.join(
    STREAMLIT_DIR,
    "app.py"
)

app_code = r'''
import os
import joblib
import numpy as np
import pandas as pd
import streamlit as st


# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

st.set_page_config(
    page_title="Predicción de rendimiento agrícola",
    page_icon="🌾",
    layout="wide"
)

BASE_DIR = os.path.dirname(
    os.path.abspath(__file__)
)


# ============================================================
# CARGAR MODELOS Y DATOS
# ============================================================

@st.cache_resource
def cargar_modelos():

    modelo_rf = joblib.load(
        os.path.join(
            BASE_DIR,
            "RF_tuned_TFM.joblib"
        )
    )

    modelo_xgb = joblib.load(
        os.path.join(
            BASE_DIR,
            "XGB_tuned_TFM.joblib"
        )
    )

    return modelo_rf, modelo_xgb


@st.cache_data
def cargar_datos():

    referencias = pd.read_csv(
        os.path.join(
            BASE_DIR,
            "referencias_provincia_cultivo_2017_2022.csv"
        )
    )

    rangos = pd.read_csv(
        os.path.join(
            BASE_DIR,
            "rangos_entrenamiento_2017_2022.csv"
        )
    )

    return referencias, rangos


modelo_rf, modelo_xgb = cargar_modelos()
referencias, rangos = cargar_datos()


# ============================================================
# VARIABLES DEL MODELO
# ============================================================

features = list(
    modelo_rf.feature_names_in_
)

num_cols = [
    c for c in features
    if c != "cultivo"
]

rangos = rangos.set_index(
    "variable"
)


# ============================================================
# CULTIVOS
# ============================================================

CULTIVOS_UI = {
    "Cebada": "barley",
    "Trigo blando": "common_soft_wheat"
}


# ============================================================
# GRUPOS DE VARIABLES
# ============================================================

variables_suelo = [
    "clay_pct_0_30cm",
    "sand_pct_0_30cm",
    "soc_g_kg_0_30cm",
    "ph_0_30cm",
    "bdod_g_cm3_0_30cm"
]

variables_sentinel = [
    c for c in num_cols
    if c.startswith(
        ("NDVI_", "EVI_", "NDMI_")
    )
]

variables_eurocrops = [
    "n_recintos",
    "superficie_eurocrops_ha",
    "superficie_media_recinto_ha"
]

variables_era5 = [
    c for c in num_cols
    if (
        c.startswith("temp_media_")
        or c.startswith("temp_rocio_")
        or c.startswith("precipitacion_")
        or c.startswith("radiacion_")
    )
]

VARS_CLIMA_RESUMEN = [
    "temp_media_c_campana",
    "temp_rocio_c_campana",
    "precipitacion_mm_campana",
    "radiacion_mj_m2_campana"
]

VARS_CLIMA_DETALLE = [
    variable
    for variable in variables_era5
    if variable not in VARS_CLIMA_RESUMEN
]

VARS_PERSONALIZABLES = (
    VARS_CLIMA_RESUMEN
    + VARS_CLIMA_DETALLE
    + variables_suelo
    + variables_sentinel
)


# ============================================================
# ETIQUETAS AMIGABLES
# ============================================================

def etiqueta_variable(variable):

    etiquetas = {
        "clay_pct_0_30cm":
            "Arcilla 0–30 cm (%)",

        "sand_pct_0_30cm":
            "Arena 0–30 cm (%)",

        "soc_g_kg_0_30cm":
            "Carbono orgánico del suelo (g/kg)",

        "ph_0_30cm":
            "pH del suelo",

        "bdod_g_cm3_0_30cm":
            "Densidad aparente (g/cm³)"
    }

    if variable in etiquetas:
        return etiquetas[variable]

    nombre = variable

    reemplazos = [
        (
            "temp_media_c_",
            "Temperatura media (°C) — "
        ),
        (
            "temp_rocio_c_",
            "Temperatura de rocío (°C) — "
        ),
        (
            "precipitacion_mm_",
            "Precipitación (mm) — "
        ),
        (
            "radiacion_mj_m2_",
            "Radiación (MJ/m²) — "
        ),
        ("NDVI_", "NDVI — "),
        ("EVI_", "EVI — "),
        ("NDMI_", "NDMI — "),
        ("ene_feb", "Enero–Febrero"),
        ("jan_feb", "Enero–Febrero"),
        ("mar_abr", "Marzo–Abril"),
        ("mar_apr", "Marzo–Abril"),
        ("may_jun", "Mayo–Junio"),
        ("jul_ago", "Julio–Agosto"),
        ("sep_dic", "Septiembre–Diciembre"),
        ("nov_dec", "Noviembre–Diciembre"),
        ("campana", "Campaña completa"),
        ("jul", "Julio")
    ]

    for antiguo, nuevo in reemplazos:
        nombre = nombre.replace(
            antiguo,
            nuevo
        )

    return nombre


# ============================================================
# FORMATO NUMÉRICO ESPAÑOL
# ============================================================

def formato_es(numero, decimales=0):

    texto = f"{numero:,.{decimales}f}"

    return (
        texto
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )


# ============================================================
# REFERENCIAS HISTÓRICAS
# ============================================================

def obtener_referencia(
    provincia,
    cultivo_modelo
):

    fila = referencias[
        (
            referencias[
                "provincia_estandar"
            ] == provincia
        )
        &
        (
            referencias[
                "cultivo"
            ] == cultivo_modelo
        )
    ]

    if fila.empty:
        return None

    return fila.iloc[0].copy()


# ============================================================
# INPUTS OPCIONALES
# ============================================================

def campo_opcional(
    variable,
    key
):
    """
    Campo vacío = utilizar referencia histórica.
    """

    valor = st.text_input(
        etiqueta_variable(variable),
        value="",
        placeholder="Usar valor histórico de referencia",
        key=key
    )

    if valor.strip() == "":
        return None

    try:
        return float(
            valor.replace(",", ".")
        )

    except ValueError:
        st.warning(
            f"Valor no válido en "
            f"'{etiqueta_variable(variable)}'. "
            f"Se utilizará la referencia histórica."
        )

        return None


# ============================================================
# CABECERA
# ============================================================

st.title(
    "🌾 Predicción de rendimiento agrícola"
)

st.subheader(
    "Demo experimental para trigo blando y cebada"
)

st.write(
    """
Para obtener una estimación básica solo es necesario indicar
el **cultivo, la provincia y la superficie de la finca**.

La aplicación completa automáticamente la información restante
con valores históricos de referencia del periodo **2017–2022**.

Si dispone de información propia sobre clima, suelo o estado
del cultivo, puede introducirla en los apartados opcionales.
Los campos que queden vacíos se completarán automáticamente.
"""
)

st.info(
    """
Esta herramienta es un prototipo demostrativo desarrollado
a partir del modelo del TFM. El modelo fue entrenado con
datos agregados a escala provincial.
"""
)


# ============================================================
# DATOS BÁSICOS
# ============================================================

st.header(
    "📍 Datos básicos de la finca"
)

col1, col2, col3 = st.columns(3)

with col1:

    cultivo_ui = st.selectbox(
        "Cultivo",
        list(
            CULTIVOS_UI.keys()
        )
    )

cultivo_modelo = (
    CULTIVOS_UI[cultivo_ui]
)

# Provincias disponibles para el cultivo seleccionado
provincias_disponibles = sorted(
    referencias.loc[
        referencias["cultivo"]
        == cultivo_modelo,
        "provincia_estandar"
    ]
    .dropna()
    .unique()
)

with col2:

    provincia = st.selectbox(
        "Provincia",
        provincias_disponibles
    )

with col3:

    superficie_ha = st.number_input(
        "Superficie de la finca (ha)",
        min_value=0.01,
        value=10.0,
        step=0.1
    )


# ============================================================
# INFORMACIÓN OPCIONAL
# ============================================================

st.markdown(
    """
### ¿Dispone de información adicional?

Los siguientes apartados son **opcionales**.

Introduzca únicamente los datos disponibles.
Los campos vacíos utilizarán automáticamente los valores
históricos de referencia.
"""
)

datos_usuario = {}


# ------------------------------------------------------------
# CLIMA GENERAL
# ------------------------------------------------------------

with st.expander(
    "🌦️ Datos meteorológicos generales — opcional"
):

    st.write(
        """
Introduzca los datos generales de la campaña si dispone
de ellos. No es necesario completar todos los campos.
"""
    )

    for variable in VARS_CLIMA_RESUMEN:

        datos_usuario[variable] = (
            campo_opcional(
                variable,
                f"general_{variable}"
            )
        )


# ------------------------------------------------------------
# CLIMA DETALLADO
# ------------------------------------------------------------

with st.expander(
    "🌦️ Meteorología detallada por periodos — avanzado"
):

    st.write(
        """
Utilice este apartado únicamente si dispone de información
meteorológica más detallada para diferentes periodos
de la campaña.
"""
    )

    for variable in VARS_CLIMA_DETALLE:

        datos_usuario[variable] = (
            campo_opcional(
                variable,
                f"meteo_{variable}"
            )
        )


# ------------------------------------------------------------
# SUELO
# ------------------------------------------------------------

with st.expander(
    "🌱 Propiedades del suelo — opcional"
):

    st.write(
        """
Si dispone de un análisis de suelo, introduzca los valores
disponibles. No es necesario completar todas las propiedades.
"""
    )

    for variable in variables_suelo:

        datos_usuario[variable] = (
            campo_opcional(
                variable,
                f"suelo_{variable}"
            )
        )


# ------------------------------------------------------------
# SATÉLITE
# ------------------------------------------------------------

with st.expander(
    "🛰️ Datos satelitales / estado del cultivo — avanzado"
):

    st.write(
        """
Introduzca estos valores únicamente si dispone de información
satelital o de índices de vegetación.

En caso contrario, deje los campos vacíos.
"""
    )

    for variable in variables_sentinel:

        datos_usuario[variable] = (
            campo_opcional(
                variable,
                f"sat_{variable}"
            )
        )


# ------------------------------------------------------------
# INFORMACIÓN TERRITORIAL
# ------------------------------------------------------------

with st.expander(
    "🗺️ Información territorial de referencia"
):

    st.write(
        """
La aplicación incorpora automáticamente información
territorial asociada a la provincia y al cultivo seleccionados.
"""
    )


# ============================================================
# BOTÓN
# ============================================================

st.header(
    "🌾 Obtener estimación"
)

calcular = st.button(
    "CALCULAR RENDIMIENTO",
    type="primary",
    use_container_width=True
)


# ============================================================
# PREDICCIÓN
# ============================================================

if calcular:

    referencia = obtener_referencia(
        provincia,
        cultivo_modelo
    )

    if referencia is None:

        st.error(
            "No existen datos de referencia para "
            "la combinación seleccionada."
        )

        st.stop()

    # --------------------------------------------------------
    # Crear entrada base con referencias
    # --------------------------------------------------------

    entrada = {}

    for variable in features:

        if variable == "cultivo":

            entrada[variable] = (
                cultivo_modelo
            )

        else:

            entrada[variable] = float(
                referencia[variable]
            )

    # --------------------------------------------------------
    # Sustituir datos personalizados
    # --------------------------------------------------------

    variables_personalizadas = []

    for variable, valor in (
        datos_usuario.items()
    ):

        if valor is not None:

            entrada[variable] = float(
                valor
            )

            variables_personalizadas.append(
                variable
            )

    X_usuario = pd.DataFrame(
        [entrada],
        columns=features
    )

    # --------------------------------------------------------
    # Modelos
    # --------------------------------------------------------

    pred_rf = float(
        modelo_rf.predict(
            X_usuario
        )[0]
    )

    pred_xgb = float(
        modelo_xgb.predict(
            X_usuario
        )[0]
    )

    pred_ensemble = (
        0.50 * pred_rf
        + 0.50 * pred_xgb
    )

    pred_ensemble = max(
        0.0,
        pred_ensemble
    )

    rendimiento_t_ha = (
        pred_ensemble / 1000
    )

    produccion_total_t = (
        pred_ensemble
        * superficie_ha
        / 1000
    )

    # --------------------------------------------------------
    # RESULTADO PRINCIPAL
    # --------------------------------------------------------

    st.success(
        "Estimación calculada correctamente"
    )

    st.subheader(
        "🌾 Rendimiento estimado"
    )

    r1, r2, r3 = st.columns(3)

    with r1:

        st.metric(
            "Rendimiento",
            f"{formato_es(pred_ensemble, 0)} kg/ha"
        )

    with r2:

        st.metric(
            "Rendimiento",
            f"{formato_es(rendimiento_t_ha, 2)} t/ha"
        )

    with r3:

        st.metric(
            "Producción total estimada",
            f"{formato_es(produccion_total_t, 2)} t"
        )

    # --------------------------------------------------------
    # DATOS UTILIZADOS
    # --------------------------------------------------------

    with st.expander(
        "📊 Ver datos utilizados para la estimación"
    ):

        n_personalizadas = len(
            variables_personalizadas
        )

        n_referencia = (
            len(num_cols)
            - n_personalizadas
        )

        if n_personalizadas == 0:

            st.write(
                """
No se han introducido datos adicionales.

La estimación utiliza valores históricos de referencia
del periodo **2017–2022** para la provincia y cultivo
seleccionados.
"""
            )

        else:

            st.write(
                f"""
**Datos introducidos por el usuario:** {n_personalizadas}

**Variables completadas automáticamente con valores
de referencia:** {n_referencia}
"""
            )

        st.markdown(
            "#### Principales valores históricos de referencia"
        )

        st.write(
            f"""
- Temperatura media de campaña:
  **{formato_es(float(referencia["temp_media_c_campana"]), 1)} °C**
- Precipitación de campaña:
  **{formato_es(float(referencia["precipitacion_mm_campana"]), 1)} mm**
- Radiación de campaña:
  **{formato_es(float(referencia["radiacion_mj_m2_campana"]), 1)} MJ/m²**
- Arena del suelo:
  **{formato_es(float(referencia["sand_pct_0_30cm"]), 1)} %**
- Arcilla del suelo:
  **{formato_es(float(referencia["clay_pct_0_30cm"]), 1)} %**
- pH del suelo:
  **{formato_es(float(referencia["ph_0_30cm"]), 2)}**
- NDVI mayo–junio:
  **{formato_es(float(referencia["NDVI_may_jun"]), 3)}**
- NDMI mayo–junio:
  **{formato_es(float(referencia["NDMI_may_jun"]), 3)}**
"""
        )

    # --------------------------------------------------------
    # CONTROL DE RANGO
    # --------------------------------------------------------

    fuera_rango = []

    for variable in num_cols:

        if variable not in rangos.index:
            continue

        valor = float(
            X_usuario.iloc[0][variable]
        )

        minimo = float(
            rangos.loc[
                variable,
                "minimo"
            ]
        )

        maximo = float(
            rangos.loc[
                variable,
                "maximo"
            ]
        )

        if (
            valor < minimo
            or valor > maximo
        ):

            fuera_rango.append(
                variable
            )

    if len(fuera_rango) == 0:

        st.success(
            "✅ Los datos utilizados están dentro del "
            "rango habitual del modelo."
        )

    else:

        st.warning(
            f"⚠️ Se han detectado {len(fuera_rango)} "
            "valores fuera del rango observado durante "
            "el entrenamiento. La estimación debe "
            "interpretarse con mayor precaución."
        )

        with st.expander(
            "Ver valores fuera del rango"
        ):

            for variable in fuera_rango:

                st.write(
                    f"- {etiqueta_variable(variable)}"
                )

    # --------------------------------------------------------
    # MODELOS
    # --------------------------------------------------------

    with st.expander(
        "🔬 Ver información del modelo"
    ):

        st.write(
            f"""
**Random Forest:** {formato_es(pred_rf, 0)} kg/ha

**XGBoost:** {formato_es(pred_xgb, 0)} kg/ha

La estimación principal combina ambos modelos
al **50 %**.
"""
        )


# ============================================================
# PIE
# ============================================================

st.divider()

st.subheader(
    "ℹ️ Alcance de la herramienta"
)

st.write(
    """
Esta demo muestra cómo puede integrarse el modelo desarrollado
en el TFM dentro de una herramienta sencilla orientada a
usuarios finales.

Para avanzar hacia una aplicación operacional sería necesario
incorporar más campañas, datos a escala de parcela y conexiones
automáticas con fuentes meteorológicas, edáficas y satelitales.
"""
)
'''

with open(
    APP_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(app_code)

print("=" * 80)
print("PASO 29 - CREACIÓN DE app.py")
print("=" * 80)

print("\nArchivo:")
print(APP_PATH)

print("\nExiste:")
print(os.path.exists(APP_PATH))

print("\nTamaño:")
print(
    round(
        os.path.getsize(APP_PATH) / 1024,
        1
    ),
    "KB"
)

print("\nPASO 29 FINALIZADO")

In [ ]:
# ============================================================
# PASO 30 - CREAR requirements.txt
# ============================================================

import os
import sys
import subprocess

# Instalar Streamlit en Colab solo para obtener
# una versión actual y dejarla fijada
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "streamlit"],
    check=True
)

import streamlit
import sklearn
import xgboost
import pandas
import numpy
import joblib

REQUIREMENTS_PATH = os.path.join(
    STREAMLIT_DIR,
    "requirements.txt"
)

requirements = f"""streamlit=={streamlit.__version__}
scikit-learn=={sklearn.__version__}
xgboost=={xgboost.__version__}
pandas=={pandas.__version__}
numpy=={numpy.__version__}
joblib=={joblib.__version__}
"""

with open(
    REQUIREMENTS_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(requirements)

print("=" * 80)
print("PASO 30 - REQUIREMENTS")
print("=" * 80)

print("\nContenido de requirements.txt:\n")
print(requirements)

print("Archivo:")
print(REQUIREMENTS_PATH)

print("\nExiste:")
print(os.path.exists(REQUIREMENTS_PATH))

print("\nPASO 30 FINALIZADO")

In [ ]:
# ============================================================
# PASO 31 - PRUEBA LOCAL DE LA APLICACIÓN STREAMLIT
# ============================================================

import os
import subprocess
import time

APP_PATH = os.path.join(
    STREAMLIT_DIR,
    "app.py"
)

# Lanzar Streamlit temporalmente
proceso_streamlit = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,
        "--server.headless=true",
        "--server.port=8501"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Dar unos segundos para que arranque
time.sleep(8)

estado = proceso_streamlit.poll()

print("=" * 80)
print("PASO 31 - PRUEBA DE ARRANQUE STREAMLIT")
print("=" * 80)

if estado is None:
    print("\nAplicación Streamlit iniciada correctamente.")
    print("No se han detectado errores de arranque.")

    # Detener el servidor de prueba
    proceso_streamlit.terminate()
    proceso_streamlit.wait(timeout=10)

    print("\nServidor de prueba detenido correctamente.")
    print("\nPASO 31 FINALIZADO")

else:
    stdout, stderr = proceso_streamlit.communicate()

    print("\nERROR AL INICIAR LA APLICACIÓN")
    print("\nSTDOUT:")
    print(stdout)

    print("\nSTDERR:")
    print(stderr)

In [ ]:
# ============================================================
# PASO 33A - COMPROBAR Y COMPRIMIR MODELOS PARA GITHUB
# ============================================================

import os
import joblib

modelos = {
    "RF_tuned_TFM.joblib":
        os.path.join(STREAMLIT_DIR, "RF_tuned_TFM.joblib"),

    "XGB_tuned_TFM.joblib":
        os.path.join(STREAMLIT_DIR, "XGB_tuned_TFM.joblib")
}

print("=" * 80)
print("PASO 33A - TAMAÑO Y COMPRESIÓN DE MODELOS")
print("=" * 80)

for nombre, ruta in modelos.items():

    tam_mb = os.path.getsize(ruta) / (1024 ** 2)

    print(f"\n{nombre}")
    print(f"Tamaño actual: {tam_mb:.2f} MB")

    # Cargar modelo
    modelo = joblib.load(ruta)

    # Guardar temporalmente con compresión
    ruta_temp = ruta + ".compressed"

    joblib.dump(
        modelo,
        ruta_temp,
        compress=3
    )

    tam_comprimido_mb = (
        os.path.getsize(ruta_temp) / (1024 ** 2)
    )

    print(
        f"Tamaño comprimido: "
        f"{tam_comprimido_mb:.2f} MB"
    )

    # Sustituir original por versión comprimida
    os.replace(
        ruta_temp,
        ruta
    )

print("\n" + "=" * 80)
print("TAMAÑOS FINALES")
print("=" * 80)

for nombre, ruta in modelos.items():

    tam_mb = os.path.getsize(ruta) / (1024 ** 2)

    estado = (
        "OK para subida web"
        if tam_mb < 25
        else "SIGUE SUPERANDO 25 MB"
    )

    print(
        f"{nombre}: {tam_mb:.2f} MB -> {estado}"
    )

print("\nPASO 33A FINALIZADO")

In [ ]:
# ============================================================
# PASO 43 - GENERAR CÓDIGO QR DE LA DEMO WEB
# ============================================================

import os
import sys
import subprocess

# Instalar librería necesaria
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "qrcode[pil]"],
    check=True
)

import qrcode

# URL pública definitiva de la aplicación
URL_DEMO = "https://tfm-rendimiento-cultivos.streamlit.app/"

# Carpeta donde guardar el QR
QR_DIR = (
    "/content/TFM_Rendimiento_cultivos/"
    "TFM_Rendimiento_cultivos/"
    "06_Demo_UI"
)

# Nombre del archivo
QR_PATH = os.path.join(
    QR_DIR,
    "QR_Demo_Prediccion_Rendimiento_TFM.png"
)

# Crear QR
qr = qrcode.QRCode(
    version=None,
    error_correction=qrcode.constants.ERROR_CORRECT_H,
    box_size=15,
    border=4
)

qr.add_data(URL_DEMO)
qr.make(fit=True)

imagen_qr = qr.make_image(
    fill_color="black",
    back_color="white"
)

# Guardar
imagen_qr.save(QR_PATH)

print("=" * 80)
print("PASO 43 - CÓDIGO QR")
print("=" * 80)

print("\nURL codificada:")
print(URL_DEMO)

print("\nArchivo:")
print(QR_PATH)

print("\nExiste:")
print(os.path.exists(QR_PATH))

print("\nPASO 43 FINALIZADO")